# Emotional KV-Cache Injection for Frozen Transformers
## A Thesis-Ready Implementation (Phi-4 Mini)

This notebook implements two variants of inference-time emotional control via KV-cache manipulation:
- **Variant 1 (Baseline)**: Simple emotion-to-KV projection
- **Variant 2 (Modulated)**: Head-wise gated modulation of KV prefixes

Three scenarios are supported:
1. **Inference-only** (pre-trained components, no further training)
2. **V1 Training** (train basic projector)
3. **V2 Training** (train projector + head-gating network)

In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from copy import deepcopy
import numpy as np
from tqdm import tqdm
import warnings

# ===================== Setup & Warnings =====================
# Suppress expandable_segments warning on unsupported platforms
warnings.filterwarnings("ignore", message=".*expandable_segments not supported.*")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = os.path.join(os.getcwd(), ".hf_cache")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(os.getcwd(), ".hf_cache", "transformers")
os.environ["HF_DATASETS_CACHE"] = os.path.join(os.getcwd(), ".hf_cache", "datasets")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

@dataclass
class KVInjectionConfig:
    """Configuration for KV injection experiments"""
    # Model names
    model_name: str = "microsoft/Phi-4-mini-instruct"
    encoder_name: str = "distilbert-base-uncased"
    
    # Dimensions
    num_emotions: int = 28
    prefix_len: int = 4
    
    # Paths
    cache_dir: str = "./.hf_cache"
    checkpoint_dir: str = "./checkpoints"
    
    # Training
    learning_rate: float = 1e-4
    batch_size: int = 4
    num_epochs: int = 2
    max_seq_len: int = 128
    
    def __post_init__(self):
        os.makedirs(self.cache_dir, exist_ok=True)
        os.makedirs(self.checkpoint_dir, exist_ok=True)

config = KVInjectionConfig()

Device: cuda


In [2]:
# ===================== ARCHITECTURE DEFINITIONS =====================

class EmotionExtractor(nn.Module):
    """
    Emotion encoder: Text → Emotion Distribution
    Outputs a probability distribution over emotional classes
    """
    def __init__(self, encoder_name=config.encoder_name, num_emotions=config.num_emotions):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, cache_dir=config.cache_dir)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_emotions)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state.mean(dim=1)  # Mean pooling
        logits = self.classifier(pooled)
        return logits  # (batch_size, num_emotions)


class BasicKVProjector(nn.Module):
    """
    VARIANT 1: Baseline KV Injection
    
    Emotion vector → Single projection → KV prefix
    All heads receive identical emotional signal
    
    IMPORTANT FIX: Uses num_kv_heads (not num_heads) for cache compatibility
    - Cache shape: (B, num_kv_heads, seq_len, head_dim)
    - Prefix shape: (B, num_kv_heads, prefix_len, head_dim)
    
    Input: emotion_vector (batch_size, num_emotions)
    Output: (k_prefix, v_prefix) for all layers
    """
    def __init__(
        self,
        num_emotions: int,
        num_heads: int,
        num_kv_heads: int,
        head_dim: int,
        prefix_len: int = config.prefix_len
    ):
        super().__init__()
        self.num_emotions = num_emotions
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.prefix_len = prefix_len
        
        # FIX: Project to num_kv_heads dimension (not num_heads)
        # This matches the KV cache shape in the decoder
        self.proj_k = nn.Linear(num_emotions, num_kv_heads * head_dim)
        self.proj_v = nn.Linear(num_emotions, num_kv_heads * head_dim)
    
    def forward(self, emotion_vec: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            emotion_vec: (batch_size, num_emotions)
        
        Returns:
            k_prefix: (batch_size, num_kv_heads, prefix_len, head_dim)
            v_prefix: (batch_size, num_kv_heads, prefix_len, head_dim)
        """
        batch_size = emotion_vec.size(0)
        
        # Project emotion to K and V dimensions
        k_proj = self.proj_k(emotion_vec)  # (batch_size, num_kv_heads * head_dim)
        v_proj = self.proj_v(emotion_vec)  # (batch_size, num_kv_heads * head_dim)
        
        # Reshape to (batch_size, num_kv_heads, 1, head_dim)
        k_prefix = k_proj.view(batch_size, self.num_kv_heads, 1, self.head_dim)
        v_prefix = v_proj.view(batch_size, self.num_kv_heads, 1, self.head_dim)
        
        # Repeat prefix_len times
        k_prefix = k_prefix.repeat(1, 1, self.prefix_len, 1)
        v_prefix = v_prefix.repeat(1, 1, self.prefix_len, 1)
        
        return k_prefix, v_prefix


class ModulatedKVProjector(nn.Module):
    """
    VARIANT 2: Head-Wise Modulated KV Injection
    
    Emotion vector → [
        KV projector (like Variant 1) +
        Head-gating network (per-head scaling)
    ]
    
    Each attention head can receive differently scaled emotional signal
    
    IMPORTANT FIX: Uses num_kv_heads for K/V projections (cache compatibility)
    """
    def __init__(
        self,
        num_emotions: int,
        num_heads: int,
        num_kv_heads: int,
        head_dim: int,
        prefix_len: int = config.prefix_len,
        gating_hidden_dim: int = 64
    ):
        super().__init__()
        self.num_emotions = num_emotions
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.prefix_len = prefix_len
        
        # === Part 1: Base KV projection (FIX: use num_kv_heads) ===
        self.proj_k = nn.Linear(num_emotions, num_kv_heads * head_dim)
        self.proj_v = nn.Linear(num_emotions, num_kv_heads * head_dim)
        
        # === Part 2: Head-gating network ===
        # Produces per-head scaling factors α_i
        # Note: This still uses num_heads for dimensionality
        self.head_gate = nn.Sequential(
            nn.Linear(num_emotions, gating_hidden_dim),
            nn.ReLU(),
            nn.Linear(gating_hidden_dim, num_heads)
        )
    
    def forward(self, emotion_vec: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            emotion_vec: (batch_size, num_emotions)
        
        Returns:
            k_prefix: (batch_size, num_kv_heads, prefix_len, head_dim)
            v_prefix: (batch_size, num_kv_heads, prefix_len, head_dim)
            head_gates: (batch_size, num_heads) - scaling factors per head
        """
        batch_size = emotion_vec.size(0)
        
        # === Base projection ===
        k_proj = self.proj_k(emotion_vec)  # (batch_size, num_kv_heads * head_dim)
        v_proj = self.proj_v(emotion_vec)  # (batch_size, num_kv_heads * head_dim)
        
        # Reshape to (batch_size, num_kv_heads, 1, head_dim)
        k_prefix = k_proj.view(batch_size, self.num_kv_heads, 1, self.head_dim)
        v_prefix = v_proj.view(batch_size, self.num_kv_heads, 1, self.head_dim)
        
        # === Head-wise gating ===
        head_gates = self.head_gate(emotion_vec)  # (batch_size, num_heads)
        head_gates = torch.sigmoid(head_gates)  # Normalize to (0, 1)
        
        # FIX: Extract only num_kv_heads gates for scaling K and V
        # This handles the case where num_heads > num_kv_heads
        head_gates_kv = head_gates[:, :self.num_kv_heads]  # (batch_size, num_kv_heads)
        
        # Apply gating: scale each KV head's prefix
        k_prefix = k_prefix * head_gates_kv.unsqueeze(-1).unsqueeze(-1)  # Broadcasting
        v_prefix = v_prefix * head_gates_kv.unsqueeze(-1).unsqueeze(-1)
        
        # Repeat prefix_len times
        k_prefix = k_prefix.repeat(1, 1, self.prefix_len, 1)
        v_prefix = v_prefix.repeat(1, 1, self.prefix_len, 1)
        
        return k_prefix, v_prefix, head_gates


# UTILITIES & HELPER FUNCTIONS

In [3]:
# ===================== HELPER FUNCTIONS =====================

def prepend_prefix_to_cache(
    past_kv: DynamicCache,
    k_prefix: torch.Tensor,
    v_prefix: torch.Tensor
) -> DynamicCache:
    """
    Prepend emotion prefix to KV cache while preserving HuggingFace DynamicCache type.
    
    This function is called ONCE before autoregressive generation.
    After this, the KV cache grows naturally with generated tokens.
    
    ⚠️  CRITICAL: We do NOT convert DynamicCache to a list. Instead:
        1. Extract (k, v) tensors from DynamicCache
        2. Prepend emotion prefix to each layer
        3. Reconstruct a new DynamicCache with the modified tensors
    
    This ensures past_key_values maintains proper cache API methods
    (e.g., get_seq_length(), which is needed by transformers).
    
    Args:
        past_kv: HuggingFace DynamicCache from model.forward(use_cache=True)
        k_prefix: (batch_size, num_heads, prefix_len, head_dim)
        v_prefix: (batch_size, num_kv_heads, prefix_len, head_dim)
    
    Returns:
        New DynamicCache with emotion prefixes prepended along sequence dimension
    """
    # Build new cache layer by layer
    new_cache = DynamicCache()
    
    # Iterate through all layers in the original cache
    for layer_idx in range(len(past_kv)):
        # Extract k, v for this layer from DynamicCache
        # Shape: (batch_size, num_heads, seq_len, head_dim)
        k_layer = past_kv[layer_idx][0]
        v_layer = past_kv[layer_idx][1]
        
        # Prepend prefix to this layer
        # k_prefix and v_prefix are broadcasted to all layers
        k_new = torch.cat(
            [k_prefix.to(k_layer.device).to(k_layer.dtype), k_layer],
            dim=2  # Concatenate along sequence dimension
        )
        v_new = torch.cat(
            [v_prefix.to(v_layer.device).to(v_layer.dtype), v_layer],
            dim=2
        )
        
        # Update cache with modified k, v for this layer
        new_cache.update(k_new, v_new, layer_idx=layer_idx)
    
    return new_cache


def get_emotion_embedding(
    text: str,
    emotion_extractor: nn.Module,
    tokenizer: AutoTokenizer,
    device: str = DEVICE
) -> torch.Tensor:
    """
    Extract emotion distribution from text.
    
    Args:
        text: Input text
        emotion_extractor: EmotionExtractor model (frozen during inference)
        tokenizer: Tokenizer for emotion encoder
        device: Device to run on
    
    Returns:
        Emotion probability distribution (1, num_emotions)
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = emotion_extractor(**inputs)
        emo_dist = torch.sigmoid(logits)  # (1, num_emotions)
    return emo_dist


def print_cache_structure(past_kv, prefix_str=""):
    """Visualize KV cache structure for debugging"""
    if not past_kv or len(past_kv) == 0:
        print(f"{prefix_str}Cache: Empty")
        return
    
    k, v = past_kv[0]
    print(f"{prefix_str}Cache structure (first layer):")
    print(f"  K shape: {k.shape}  (batch, heads, seq_len, head_dim)")
    print(f"  V shape: {v.shape}  (batch, heads, seq_len, head_dim)")
    print(f"  Total layers: {len(past_kv)}")


# ===================== DATASET =====================

class EmotionalResponseDataset(Dataset):
    """
    Dataset for training emotion-conditioned text generation.
    
    Pairs: (input_text, target_response)
    Emotions are extracted from input_text via EmotionExtractor
    """
    def __init__(
        self,
        pairs: List[Dict[str, str]],
        tokenizer: AutoTokenizer,
        emotion_extractor: nn.Module,
        encoder_tokenizer: AutoTokenizer,
        max_len: int = 128,
        device: str = DEVICE
    ):
        """
        Args:
            pairs: List of {"input": "...", "output": "..."}
            tokenizer: Tokenizer for decoder (Phi-4, etc.)
            emotion_extractor: Pre-trained emotion encoder
            encoder_tokenizer: Tokenizer for emotion encoder (DistilBERT, etc.)
            max_len: Maximum sequence length
            device: Device to extract emotions on
        """
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.emotion_extractor = emotion_extractor
        self.encoder_tokenizer = encoder_tokenizer
        self.max_len = max_len
        self.device = device
        
        # Pre-compute emotion embeddings
        self.emotions = []
        emotion_extractor.eval()
        with torch.no_grad():
            for pair in pairs:
                emo = get_emotion_embedding(
                    pair["input"],
                    emotion_extractor,
                    encoder_tokenizer,
                    device
                )
                self.emotions.append(emo.cpu())

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        
        # Encode input and output
        input_enc = self.tokenizer(
            pair["input"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        output_enc = self.tokenizer(
            pair["output"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": input_enc["input_ids"].squeeze(0),
            "attention_mask": input_enc["attention_mask"].squeeze(0),
            "labels": output_enc["input_ids"].squeeze(0),
            "emotion": self.emotions[idx].squeeze(0)
        }


def compute_emotional_alignment(
    input_text: str,
    vanilla_output: str,
    v1_output: str,
    v2_output: str,
    emotion_extractor: nn.Module,
    encoder_tokenizer: AutoTokenizer,
    device: str = DEVICE
) -> dict[str, list[float]]:
    """
    Evaluate emotional alignment: how well does the output preserve the input emotion?
    
    Computes cosine similarity between:
    - Input emotion vector and vanilla output emotion vector
    - Input emotion vector and KV-injected output emotion vector
    
    Args:
        input_text: Input prompt text
        vanilla_output: Vanilla decoder output
        kv_output: KV-injected decoder output
        emotion_extractor: EmotionExtractor model (frozen, DistilBERT-based)
        encoder_tokenizer: Tokenizer for emotion extractor
        device: Device to compute on
    
    Returns:
        sim_vanilla: Cosine similarity between input and vanilla output emotions (0-1)
        sim_kv: Cosine similarity between input and KV output emotions (0-1)
        delta: Improvement (sim_kv - sim_vanilla)
    """
    emotion_extractor.eval()
    
    with torch.no_grad():
        # Extract emotion vectors for all three texts
        e_in = get_emotion_embedding(input_text, emotion_extractor, encoder_tokenizer, device)
        e_vanilla = get_emotion_embedding(vanilla_output, emotion_extractor, encoder_tokenizer, device)
        e_v1 = get_emotion_embedding(v1_output, emotion_extractor, encoder_tokenizer, device)
        e_v2 = get_emotion_embedding(v2_output, emotion_extractor, encoder_tokenizer, device)

        
        # Move to CPU for cosine similarity computation
        e_in = e_in.cpu().squeeze(0)  # (28,)
        e_vanilla = e_vanilla.cpu().squeeze(0)  # (28,)
        e_v1 = e_v1.cpu().squeeze(0)  # (28,)
        e_v2 = e_v2.cpu().squeeze(0)  # (28,)
        
        # Compute cosine similarity: dot(a,b) / (||a|| * ||b||)
        # For normalized vectors, this is just the dot product
        sim_vanilla = torch.nn.functional.cosine_similarity(
            e_in.unsqueeze(0), 
            e_vanilla.unsqueeze(0)
        ).item()
        
        sim_v1 = torch.nn.functional.cosine_similarity(
            e_in.unsqueeze(0), 
            e_v1.unsqueeze(0)
        ).item()
        
        sim_v2 = torch.nn.functional.cosine_similarity(
            e_in.unsqueeze(0), 
            e_v2.unsqueeze(0)
        ).item()
        
        
        delta_v1 = sim_v1 - sim_vanilla
        delta_v2 = sim_v2 - sim_vanilla
        
        values = {
            "sim_values" : [sim_v1, sim_v2],
            "delta_values" : [delta_v1, delta_v2]
        }
    
    return values

# TRAINING: VARIANT 1 (BasicKVProjector)

In [4]:
# ===================== TRAINING: VARIANT 1 =====================

def train_variant_1(
    train_pairs: List[Dict[str, str]],
    val_pairs: List[Dict[str, str]] = None,
    num_epochs: int = config.num_epochs,
    batch_size: int = config.batch_size,
    learning_rate: float = 1e-4,
    save_path: str = None
):
    """
    Train BasicKVProjector (Variant 1).
    
    FROZEN: Decoder, Emotion Extractor
    TRAINABLE: Basic K/V projector only
    """
    print("=" * 70)
    print("TRAINING VARIANT 1: BasicKVProjector")
    print("=" * 70)
    
    if save_path is None:
        save_path = os.path.join(config.checkpoint_dir, "variant1_projector.pt")
    
    # Load models
    print("\n[1/4] Loading models...")
    
    # Decoder (frozen)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    decoder_tokenizer = AutoTokenizer.from_pretrained(config.model_name, cache_dir=config.cache_dir)
    if decoder_tokenizer.pad_token is None:
        decoder_tokenizer.pad_token = decoder_tokenizer.eos_token
    
    decoder = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        cache_dir=config.cache_dir,
        device_map="auto",
        quantization_config=bnb_config,
        dtype=torch.float16
    )
    decoder.eval()
    
    # Emotion extractor (frozen)
    encoder_tokenizer = AutoTokenizer.from_pretrained(config.encoder_name, cache_dir=config.cache_dir)
    emotion_extractor = EmotionExtractor().to(DEVICE).eval()
    
    # Try to load pre-trained emotion extractor
    try:
        emo_checkpoint = torch.load("checkpoint.pt", map_location="cpu")
        emotion_extractor.load_state_dict(emo_checkpoint['model_state_dict'])
        print("  ✓ Emotion extractor loaded from checkpoint.pt")
    except FileNotFoundError:
        print("  ⚠ Using untrained emotion extractor (checkpoint.pt not found)")
    
    # Initialize projector (trainable)
    projector = BasicKVProjector(
        num_emotions=config.num_emotions,
        num_heads=decoder.config.num_attention_heads,
        num_kv_heads=decoder.config.num_key_value_heads,
        head_dim=decoder.config.hidden_size // decoder.config.num_attention_heads,
        prefix_len=config.prefix_len
    ).to(DEVICE)
    print(f"  ✓ BasicKVProjector initialized with {sum(p.numel() for p in projector.parameters()):,} parameters")
    
    # Create dataset
    print("\n[2/4] Creating dataset...")
    train_dataset = EmotionalResponseDataset(
        train_pairs,
        decoder_tokenizer,
        emotion_extractor,
        encoder_tokenizer,
        max_len=config.max_seq_len,
        device=DEVICE
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    if val_pairs:
        val_dataset = EmotionalResponseDataset(
            val_pairs,
            decoder_tokenizer,
            emotion_extractor,
            encoder_tokenizer,
            max_len=config.max_seq_len,
            device=DEVICE
        )
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    else:
        val_loader = None
    
    print(f"  ✓ Train samples: {len(train_dataset)}")
    if val_loader:
        print(f"  ✓ Val samples: {len(val_dataset)}")
    
    # Training setup
    print("\n[3/4] Setting up training...")
    optimizer = optim.Adam(projector.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss(ignore_index=decoder_tokenizer.pad_token_id)
    scaler = torch.amp.GradScaler(device=DEVICE)  # For mixed precision training
    
    best_val_loss = float('inf')
    training_history = {
        'train_loss': [],
        'val_loss': []
    }
    
    # Training loop
    print("\n[4/4] Training...\n")
    for epoch in tqdm(range(num_epochs), desc="Epochs for vairant 1"):
        # Training phase
        projector.train()
        train_loss = 0.0
        
        for batch_idx, batch in tqdm(enumerate(train_loader), desc = "Batches for variant 1"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            emotions = batch["emotion"].to(DEVICE)
            
            # Forward pass through frozen decoder
            with torch.no_grad():
                out = decoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=True
                )
                past_kv = out.past_key_values  # DynamicCache object
            
            # Get emotion-based KV prefix (with mixed precision)
            with torch.amp.autocast(device_type=DEVICE):
                k_prefix, v_prefix = projector(emotions)
            
            # Prepend prefix to cache (preserves DynamicCache type)
            past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
            
            # Forward again with modified cache (with mixed precision)
            with torch.amp.autocast(device_type=DEVICE):
                logits = decoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    past_key_values=past_kv_modified,
                    use_cache=False
                ).logits
                loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
                
            scaler.scale(loss).backward()
            
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            
            if (batch_idx + 1) % max(1, len(train_loader) // 5) == 0:
                print(f"  Epoch {epoch+1}/{num_epochs} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
        # Clear GPU cache after each epoch
        torch.cuda.empty_cache()
        
        avg_train_loss = train_loss / len(train_loader)
        training_history['train_loss'].append(avg_train_loss)
        
        # Validation phase
        if val_loader:
            projector.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(DEVICE)
                    attention_mask = batch["attention_mask"].to(DEVICE)
                    labels = batch["labels"].to(DEVICE)
                    emotions = batch["emotion"].to(DEVICE)
                    
                    out = decoder(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        use_cache=True
                    )
                    # past_kv is a DynamicCache - do NOT convert to list
                    past_kv = out.past_key_values
                    
                    k_prefix, v_prefix = projector(emotions)
                    past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
                    
                    logits = decoder(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        past_key_values=past_kv_modified,
                        use_cache=False
                    ).logits
                    
                    loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
                    val_loss += loss.item()
            
            avg_val_loss = val_loss / len(val_loader)
            training_history['val_loss'].append(avg_val_loss)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                print(f"\n  ✓ New best val loss: {avg_val_loss:.4f}")
                # Save checkpoint
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': projector.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': avg_train_loss,
                    'val_loss': avg_val_loss,
                }, save_path)
                print(f"  ✓ Checkpoint saved to {save_path}\n")
        else:
            print(f"\nEpoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f}")
            # Save checkpoint even without validation
            torch.save({
                'epoch': epoch,
                'model_state_dict': projector.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
            }, save_path)
    print("\n" + "=" * 70)
    print("\n" + "=" * 70)
    print(f"Training complete! Checkpoint saved to {save_path}")
    print("=" * 70)
    
    # Save training history to CSV
    import pandas as pd
    df_history = pd.DataFrame(training_history)
    csv_path = os.path.join(config.checkpoint_dir, 'variant1_training_history.csv')

    df_history.to_csv(csv_path, index=False)
    print(f"✓ Training history saved to {csv_path}")    

    return projector, training_history
    

In [5]:
# ===================== TRAINING: VARIANT 2 =====================
def train_variant_2(
    train_pairs: List[Dict[str, str]],
    val_pairs: List[Dict[str, str]] = None,
    num_epochs: int = config.num_epochs,
    batch_size: int = config.batch_size,
    learning_rate: float = 1e-4,
    save_path: str = None,
    gating_hidden_dim: int = 64,
    variant1_checkpoint: str = None
):
    """
    Train ModulatedKVProjector (Variant 2).
    
    FROZEN: Decoder, Emotion Extractor, (optionally) Variant 1 base projector
    TRAINABLE: Head-gating network ONLY (if variant1_checkpoint provided), or both base+gating
    
    This extends Variant 1 by learning per-head scaling factors.
    If variant1_checkpoint is provided, loads its weights and freezes them.
    """
    print("=" * 70)
    print("TRAINING VARIANT 2: ModulatedKVProjector (with Head-Gating)")
    print("=" * 70)
    
    if save_path is None:
        save_path = os.path.join(config.checkpoint_dir, "variant2_projector.pt")
    
    # Load models
    print("\n[1/4] Loading models...")
    
    # Decoder (frozen)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    decoder_tokenizer = AutoTokenizer.from_pretrained(config.model_name, cache_dir=config.cache_dir)
    if decoder_tokenizer.pad_token is None:
        decoder_tokenizer.pad_token = decoder_tokenizer.eos_token
    
    decoder = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        cache_dir=config.cache_dir,
        device_map="auto",
        quantization_config=bnb_config,
        dtype=torch.float16
    )
    decoder.eval()
    
    # Emotion extractor (frozen)
    encoder_tokenizer = AutoTokenizer.from_pretrained(config.encoder_name, cache_dir=config.cache_dir)
    emotion_extractor = EmotionExtractor().to(DEVICE).eval()
    
    try:
        emo_checkpoint = torch.load("checkpoint.pt", map_location="cpu")
        emotion_extractor.load_state_dict(emo_checkpoint['model_state_dict'])
        print("  ✓ Emotion extractor loaded from checkpoint.pt")
    except FileNotFoundError:
        print("  ⚠ Using untrained emotion extractor (checkpoint.pt not found)")
    
    # Initialize projector with head-gating (trainable)
    projector = ModulatedKVProjector(
        num_emotions=config.num_emotions,
        num_heads=decoder.config.num_attention_heads,
        num_kv_heads=decoder.config.num_key_value_heads,
        head_dim=decoder.config.hidden_size // decoder.config.num_attention_heads,
        prefix_len=config.prefix_len,
        gating_hidden_dim=gating_hidden_dim
    ).to(DEVICE)
    
    v1pth = os.path.join(config.checkpoint_dir, "variant1_projector.pt")
    # Load and freeze Variant 1 weights if provided
    if v1pth and os.path.exists(v1pth):
        print(f"\n  Loading Variant 1 weights from {v1pth}...")
        checkpoint = torch.load(v1pth, map_location="cpu")
        
        # Load only the base projection weights (proj_k, proj_v)
        variant1_state = {k: v for k, v in checkpoint['model_state_dict'].items() 
                         if k.startswith('proj_')}
        
        projector.load_state_dict(variant1_state, strict=False)
        print(f"  ✓ Loaded {len(variant1_state)} Variant 1 layers")
        
        # FREEZE Variant 1 weights
        for name, param in projector.named_parameters():
            if name.startswith('proj_'):
                param.requires_grad = False
    
    # Count parameters
    trainable_params = sum(p.numel() for p in projector.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in projector.parameters() if not p.requires_grad)
    total_params = sum(p.numel() for p in projector.parameters())
    
    print(f"  ✓ ModulatedKVProjector initialized with {total_params:,} parameters")
    print(f"    - Base projector (proj_k, proj_v): {frozen_params:,} {'(frozen)' if frozen_params > 0 else '(trainable)'}") 
    print(f"    - Head-gating network: {trainable_params:,} (trainable)")
    
    # Create dataset
    print("\n[2/4] Creating dataset...")
    train_dataset = EmotionalResponseDataset(
        train_pairs,
        decoder_tokenizer,
        emotion_extractor,
        encoder_tokenizer,
        max_len=config.max_seq_len,
        device=DEVICE
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    if val_pairs:
        val_dataset = EmotionalResponseDataset(
            val_pairs,
            decoder_tokenizer,
            emotion_extractor,
            encoder_tokenizer,
            max_len=config.max_seq_len,
            device=DEVICE
        )
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    else:
        val_loader = None
    
    print(f"  ✓ Train samples: {len(train_dataset)}")
    if val_loader:
        print(f"  ✓ Val samples: {len(val_dataset)}")
    
    # Training setup
    print("\n[3/4] Setting up training...")
    # Only optimize trainable parameters
    trainable_params_list = [p for p in projector.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params_list, lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss(ignore_index=decoder_tokenizer.pad_token_id)
    scaler = torch.amp.GradScaler(device=DEVICE)  # For mixed precision training
    
    best_val_loss = float('inf')
    training_history = {
        'train_loss': [],
        'val_loss': [],
        'gating_entropy': []  # Track how diverse head gates are
    }
    
    # Training loop
    print("\n[4/4] Training...\n")
    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        # Training phase
        projector.train()
        train_loss = 0.0
        gating_entropy_sum = 0.0
        
        for batch_idx, batch in tqdm(enumerate(train_loader), desc = "Batches for variant 1"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            emotions = batch["emotion"].to(DEVICE)
            
            # Forward pass through frozen decoder
            with torch.no_grad():
                out = decoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=True
                )
                past_kv = out.past_key_values
            
            # Get emotion-based KV prefix WITH head gating (with mixed precision)
            with torch.amp.autocast(device_type=DEVICE):
                k_prefix, v_prefix, head_gates = projector(emotions)
            
            # Prepend prefix to cache (preserves DynamicCache type)
            past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
            
            # Forward with modified cache (with mixed precision)
            with torch.amp.autocast(device_type=DEVICE):
                logits = decoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    past_key_values=past_kv_modified,
                    use_cache=False
                ).logits
                
                loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
                
                # Optional: Regularization on head gates (encourage diversity)
                # This prevents all heads from receiving identical scaling
                gate_entropy = -(head_gates * torch.log(head_gates + 1e-8) + 
                               (1 - head_gates) * torch.log(1 - head_gates + 1e-8)).mean()
                loss = loss + 0.01 * gate_entropy  # Weak regularization
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            gating_entropy_sum += gate_entropy.item()
            
            if (batch_idx + 1) % max(1, len(train_loader) // 5) == 0:
                print(f"  Epoch {epoch+1}/{num_epochs} | Batch {batch_idx+1}/{len(train_loader)} | "
                      f"Loss: {loss.item():.4f} | Gate Entropy: {gate_entropy.item():.4f}")
        
        # Clear GPU cache after each epoch
        torch.cuda.empty_cache()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_gating_entropy = gating_entropy_sum / len(train_loader)
        training_history['train_loss'].append(avg_train_loss)
        training_history['gating_entropy'].append(avg_gating_entropy)
        
        # Validation phase
        if val_loader:
            projector.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(DEVICE)
                    attention_mask = batch["attention_mask"].to(DEVICE)
                    labels = batch["labels"].to(DEVICE)
                    emotions = batch["emotion"].to(DEVICE)
                    
                    out = decoder(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        use_cache=True
                    )
                    # past_kv is a DynamicCache - do NOT convert to list
                    past_kv = out.past_key_values
                    
                    k_prefix, v_prefix, _ = projector(emotions)
                    past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
                    
                    logits = decoder(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        past_key_values=past_kv_modified,
                        use_cache=False
                    ).logits
                    
                    loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
                    val_loss += loss.item()
            
            avg_val_loss = val_loss / len(val_loader)
            training_history['val_loss'].append(avg_val_loss)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                print(f"\n  ✓ New best val loss: {avg_val_loss:.4f}")
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': projector.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': avg_train_loss,
                    'val_loss': avg_val_loss,
                }, save_path)
                print(f"  ✓ Checkpoint saved to {save_path}\n")
        else:
            print(f"\nEpoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | "
                  f"Gate Entropy: {avg_gating_entropy:.4f}")
            torch.save({
                'epoch': epoch,
                'model_state_dict': projector.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
            }, save_path)
    print("=" * 70)
    print("\n" + "=" * 70)
    print(f"Training complete! Checkpoint saved to {save_path}")
    print("=" * 70)
    
    # Save training history to CSV
    import pandas as pd
    df_history = pd.DataFrame(training_history)
    csv_path = os.path.join(config.checkpoint_dir, 'variant2_training_history.csv')

    df_history.to_csv(csv_path, index=False)
    print(f"✓ Training history saved to {csv_path}")
    return projector, training_history

In [5]:
# ===================== INFERENCE ENGINE =====================

class KVInjectionInference:
    """
    Unified inference interface for all variants and scenarios.
    
    Supports:
    - Scenario 1: Baseline (no injection, vanilla generation)
    - Scenario 2: Variant 1 inference (with trained/untrained BasicKVProjector)
    - Scenario 3: Variant 2 inference (with trained/untrained ModulatedKVProjector)
    """
    
    def __init__(
        self,
        decoder_name: str = config.model_name,
        encoder_name: str = config.encoder_name,
        device: str = DEVICE
    ):
        print("Initializing KV Injection Inference Engine...")
        
        self.device = device
        
        # Load decoder
        print("\n  Loading decoder...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        self.decoder_tokenizer = AutoTokenizer.from_pretrained(
            decoder_name,
            cache_dir=config.cache_dir
        )
        if self.decoder_tokenizer.pad_token is None:
            self.decoder_tokenizer.pad_token = self.decoder_tokenizer.eos_token
        
        self.decoder = AutoModelForCausalLM.from_pretrained(
            decoder_name,
            cache_dir=config.cache_dir,
            device_map="auto",
            quantization_config=bnb_config,
            dtype=torch.float16
        )
        self.decoder.eval()
        print(f"  ✓ Decoder: {decoder_name}")
        
        # Load emotion extractor
        print("  Loading emotion extractor...")
        self.encoder_tokenizer = AutoTokenizer.from_pretrained(
            encoder_name,
            cache_dir=config.cache_dir
        )
        self.emotion_extractor = EmotionExtractor().to(device).eval()
        
        try:
            emo_checkpoint = torch.load("checkpoint.pt", map_location="cpu")
            self.emotion_extractor.load_state_dict(emo_checkpoint['model_state_dict'])
            print(f"  ✓ Emotion extractor: {encoder_name} (pre-trained)")
        except FileNotFoundError:
            print(f"  ⚠ Emotion extractor: {encoder_name} (untrained)")
        
        # Projectors (will be initialized on demand)
        self.projector_v1 = None
        self.projector_v2 = None
    
    def load_projector_v1(self, checkpoint_path: str):
        """Load trained Variant 1 projector"""
        self.projector_v1 = BasicKVProjector(
            num_emotions=config.num_emotions,
            num_heads=self.decoder.config.num_attention_heads,
            num_kv_heads=self.decoder.config.num_key_value_heads,
            head_dim=self.decoder.config.hidden_size // self.decoder.config.num_attention_heads,
            prefix_len=config.prefix_len
        ).to(self.device).eval()
        
        try:
            checkpoint = torch.load(checkpoint_path, map_location="cpu")
            self.projector_v1.load_state_dict(checkpoint['model_state_dict'])
            print(f"✓ Variant 1 projector loaded from {checkpoint_path}")
            return True
        except FileNotFoundError:
            print(f"⚠ Variant 1 checkpoint not found at {checkpoint_path}")
            return False
    
    def load_projector_v2(self, checkpoint_path: str):
        """Load trained Variant 2 projector"""
        self.projector_v2 = ModulatedKVProjector(
            num_emotions=config.num_emotions,
            num_heads=self.decoder.config.num_attention_heads,
            num_kv_heads=self.decoder.config.num_key_value_heads,
            head_dim=self.decoder.config.hidden_size // self.decoder.config.num_attention_heads,
            prefix_len=config.prefix_len
        ).to(self.device).eval()
        
        try:
            checkpoint = torch.load(checkpoint_path, map_location="cpu")
            self.projector_v2.load_state_dict(checkpoint['model_state_dict'])
            print(f"✓ Variant 2 projector loaded from {checkpoint_path}")
            return True
        except FileNotFoundError:
            print(f"⚠ Variant 2 checkpoint not found at {checkpoint_path}")
            return False
    
    def generate_vanilla(
        self,
        prompt: str,
        max_new_tokens: int = 100,
        temperature: float = 0.7,
        top_p: float = 0.95
    ) -> str:
        """
        SCENARIO 1: Baseline generation (no KV injection)
        """
        inputs = self.decoder_tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output = self.decoder.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=self.decoder_tokenizer.eos_token_id
            )
        return self.decoder_tokenizer.decode(output[0], skip_special_tokens=True)
    
    def generate_with_variant_1(
        self,
        prompt: str,
        emotion_text: str,
        max_new_tokens: int = 100,
        temperature: float = 0.7,
        top_p: float = 0.95,
        show_cache_info: bool = False,
        device = DEVICE
    ) -> Tuple[str, Dict]:
        """
        SCENARIO 2: Variant 1 KV Injection
        
        Uses BasicKVProjector (single projection, all heads uniform)
        """
        if self.projector_v1 is None:
            raise ValueError("Variant 1 projector not loaded. Call load_projector_v1() first.")
        
        # Extract emotion from context
        emotion_vec = get_emotion_embedding(
            emotion_text,
            self.emotion_extractor,
            self.encoder_tokenizer,
            self.device
        )
        
        # Initial forward to get cache
        inputs = self.decoder_tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.decoder(
                **inputs,
                use_cache=True
            )
            past_kv = out.past_key_values  # Keep as DynamicCache (NOT a list)
        
        if show_cache_info:
            print("\n📊 CACHE STRUCTURE (before injection):")
            print_cache_structure(past_kv)
        
        # Get KV prefix from emotion
        with torch.no_grad():
            k_prefix, v_prefix = self.projector_v1(emotion_vec)
        
        # Prepend prefix
        past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
        
        if show_cache_info:
            print("\n📊 CACHE STRUCTURE (after injection):")
            print_cache_structure(past_kv_modified)
            print(f"\n📌 Emotion vector: {emotion_vec[0][:5].tolist()}... (first 5 of {emotion_vec.size(-1)})")
            print(f"📌 K prefix shape: {k_prefix.shape}")
            print(f"📌 V prefix shape: {v_prefix.shape}\n")
        
        # Generate with modified cache
        generated = inputs["input_ids"]
        with torch.no_grad():
            for _ in range(max_new_tokens):
                out = self.decoder(
                    input_ids=generated[:, -1:],
                    use_cache=True,
                    past_key_values=past_kv_modified
                )
                logits = out.logits[:, -1, :]
                past_kv_modified = out.past_key_values
                
                probs = torch.softmax(logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                generated = torch.cat([generated, next_token], dim=-1)
                
                if next_token.item() == self.decoder_tokenizer.eos_token_id:
                    break
        
        output_text = self.decoder_tokenizer.decode(generated[0], skip_special_tokens=True)
        
        debug_info = {
            "variant": "Variant 1 (BasicKVProjector)",
            "emotion_vector": emotion_vec[0].detach().cpu().numpy(),
            "cache_prefix_len": config.prefix_len,
            "num_layers": len(past_kv)
        }
        
        return output_text, debug_info
    
    def generate_with_variant_2(
        self,
        prompt: str,
        emotion_text: str,
        max_new_tokens: int = 100,
        temperature: float = 0.7,
        top_p: float = 0.95,
        show_cache_info: bool = False
    ) -> Tuple[str, Dict]:
        """
        SCENARIO 3: Variant 2 KV Injection with Head-Wise Modulation
        
        Uses ModulatedKVProjector (projection + per-head gating)
        """
        if self.projector_v2 is None:
            raise ValueError("Variant 2 projector not loaded. Call load_projector_v2() first.")
        
        # Extract emotion
        emotion_vec = get_emotion_embedding(
            emotion_text,
            self.emotion_extractor,
            self.encoder_tokenizer,
            self.device
        )
        
        # Initial forward
        inputs = self.decoder_tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.decoder(
                **inputs,
                use_cache=True
            )
            past_kv = out.past_key_values  # Keep as DynamicCache (NOT a list)
        
        if show_cache_info:
            print("\n📊 CACHE STRUCTURE (before injection):")
            print_cache_structure(past_kv)
        
        # Get KV prefix with head gating
        with torch.no_grad():
            k_prefix, v_prefix, head_gates = self.projector_v2(emotion_vec)
        
        # Prepend prefix
        past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
        
        if show_cache_info:
            print("\n📊 CACHE STRUCTURE (after injection):")
            print_cache_structure(past_kv_modified)
            print(f"\n📌 Emotion vector: {emotion_vec[0][:5].tolist()}... (first 5 of {emotion_vec.size(-1)})")
            print(f"📌 Head gates: {head_gates[0][:8].tolist()}... (first 8 of {head_gates.size(-1)})")
            print(f"📌 K prefix shape: {k_prefix.shape}")
            print(f"📌 V prefix shape: {v_prefix.shape}\n")
        
        # Generate with modified cache
        generated = inputs["input_ids"]
        with torch.no_grad():
            for _ in range(max_new_tokens):
                out = self.decoder(
                    input_ids=generated[:, -1:],
                    use_cache=True,
                    past_key_values=past_kv_modified
                )
                logits = out.logits[:, -1, :]
                past_kv_modified = out.past_key_values
                
                probs = torch.softmax(logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                generated = torch.cat([generated, next_token], dim=-1)
                
                if next_token.item() == self.decoder_tokenizer.eos_token_id:
                    break
        
        output_text = self.decoder_tokenizer.decode(generated[0], skip_special_tokens=True)
        
        debug_info = {
            "variant": "Variant 2 (ModulatedKVProjector)",
            "emotion_vector": emotion_vec[0].detach().cpu().numpy(),
            "head_gates": head_gates[0].detach().cpu().numpy(),
            "cache_prefix_len": config.prefix_len,
            "num_layers": len(past_kv),
            "num_heads": self.decoder.config.num_attention_heads
        }
        
        return output_text, debug_info

In [7]:
import matplotlib.pyplot as plt

def plot_training_history(
    history: dict,
    title: str = "Training History",
    show: bool = True,
    save_path: str = None
):
    """
    Plot training and validation loss curves.

    Args:
        history: dict with keys 'train_loss' and optionally 'val_loss'
        title: Title of the plot
        show: Whether to display the plot
        save_path: If provided, saves the figure to this path
    """
    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])

    epochs = range(1, len(train_loss) + 1)

    plt.figure(figsize=(8, 5))
    
    # Train loss
    plt.plot(
        epochs,
        train_loss,
        marker="o",
        linestyle="-",
        linewidth=2,
        label="Train Loss"
    )

    # Validation loss (if exists)
    if len(val_loss) > 0:
        plt.plot(
            epochs,
            val_loss,
            marker="s",
            linestyle="--",
            linewidth=2,
            label="Validation Loss"
        )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"📁 Loss plot saved to: {save_path}")

    if show:
        plt.show()
    else:
        plt.close()

# ⚠️  IMPORTANT FIX: DynamicCache Handling in KV-Prefix Injection

## The Problem (AttributeError: 'list' object has no attribute 'get_seq_length')

**What was happening:**
```python
out = self.decoder(**inputs, use_cache=True)
past_kv = list(out.past_key_values)  # ❌ WRONG: Converts DynamicCache to list
past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
decoder(..., past_key_values=past_kv_modified)  # Fails: list has no get_seq_length()
```

**Why it fails:**
- HuggingFace's `DynamicCache` is a smart object with methods like `get_seq_length()`, `update()`, etc.
- Converting to a Python list loses all these methods
- When the decoder tries to use the cache, it calls `get_seq_length()` → AttributeError on a plain list

---

## The Solution

**Correct approach:**
```python
out = self.decoder(**inputs, use_cache=True)
past_kv = out.past_key_values  # ✅ CORRECT: Keep as DynamicCache
past_kv_modified = prepend_prefix_to_cache(past_kv, k_prefix, v_prefix)
decoder(..., past_key_values=past_kv_modified)  # Works: DynamicCache has all methods
```

---

## How prepend_prefix_to_cache Works Now

```python
def prepend_prefix_to_cache(
    past_kv: DynamicCache,  # Input is DynamicCache (NOT a list)
    k_prefix: torch.Tensor,
    v_prefix: torch.Tensor
) -> DynamicCache:  # Returns DynamicCache (NOT a list)
    
    # Create new DynamicCache to hold modified tensors
    new_cache = DynamicCache()
    
    # For each layer in the transformer
    for layer_idx in range(len(past_kv)):
        # Extract K, V for this layer
        k_layer = past_kv[layer_idx][0]  # (batch, heads, seq_len, head_dim)
        v_layer = past_kv[layer_idx][1]
        
        # Prepend emotion prefix along sequence dimension
        k_new = torch.cat([k_prefix.to(k_layer.device).to(k_layer.dtype), k_layer], dim=2)
        v_new = torch.cat([v_prefix.to(v_layer.device).to(v_layer.dtype), v_layer], dim=2)
        
        # Update new cache (preserves DynamicCache structure)
        new_cache.update(k_new, v_new, layer_idx=layer_idx)
    
    return new_cache  # Valid DynamicCache with all methods intact
```

---

## Key Changes Made

| Location | Before | After |
|----------|--------|-------|
| `train_variant_1` (train phase) | `past_kv = list(out.past_key_values)` | `past_kv = out.past_key_values` |
| `train_variant_1` (val phase) | `past_kv = list(out.past_key_values)` | `past_kv = out.past_key_values` |
| `train_variant_2` (train phase) | `past_kv_list = list(past_kv)` + legacy conversion | Direct `prepend_prefix_to_cache(past_kv, ...)` |
| `train_variant_2` (val phase) | `past_kv = list(out.past_key_values)` | `past_kv = out.past_key_values` |
| `generate_with_variant_1` | `past_kv = list(out.past_key_values)` | `past_kv = out.past_key_values` |
| `generate_with_variant_2` | `past_kv = list(out.past_key_values)` | `past_kv = out.past_key_values` |

---

## Why This Matters

1. **Preserves Cache API**: DynamicCache has methods the decoder depends on
2. **Cleaner Code**: No list conversions, simpler logic
3. **Future-Proof**: Works with any HuggingFace transformers version
4. **Correct Semantics**: DynamicCache is designed for this use case

---

## Sigmoid Usage (NOT Changed)

The sigmoid in two places is correct and intentional:

### 1. Emotion Extractor (Correct ✓)
```python
def get_emotion_embedding(...):
    logits = emotion_extractor(**inputs)
    emo_dist = torch.sigmoid(logits)  # Multi-label classification
    return emo_dist
```
**Why:** Emotion classification is multi-label (multiple emotions can co-occur), so sigmoid is correct.

### 2. Head-Wise Gating in Variant 2 (Correct ✓)
```python
self.head_gate = nn.Sequential(
    nn.Linear(...),
    nn.ReLU(),
    nn.Linear(..., num_heads)
)
# In forward:
head_gates = torch.sigmoid(self.head_gate(emotion_vec))  # α ∈ (0, 1)
```
**Why:** We want per-head scaling factors in [0, 1], sigmoid does this.

### Decoder Loss (Correct ✓)
```python
loss = nn.CrossEntropyLoss()(logits, labels)
```
**Why:** CrossEntropyLoss expects raw logits (NO softmax applied manually). This is standard.

No changes needed here - sigmoid usage is mathematically correct for the task.

---

## Testing the Fix

```python
# This should now work without AttributeError
engine = KVInjectionInference()
output, debug_info = engine.generate_with_variant_1(
    prompt="Test prompt",
    emotion_text="happy"
)
print(output)  # ✓ Should work
```

In [8]:
# Example dataset for thesis
example_pairs = [
    # TRUST & BETRAYAL (1-15)
    {
        "input": "My friend betrayed my trust and I feel devastated.",
        "output": "That's a painful experience. Betrayal can deeply hurt, but remember that this doesn't define your worth. Consider taking time to process your emotions and decide what boundaries feel right for you going forward."
    },
    { "input": "I trusted someone completely and they lied to me.", "output": "That broken trust feels like a deep wound. It's natural to feel angry and hurt. Take time to rebuild your ability to trust—in others and in yourself." },
    { "input": "My family kept secrets from me and I feel excluded.", "output": "Feeling left out of family matters can sting. When you're ready, open communication about expectations might help heal things." },
    { "input": "A colleague threw me under the bus at work.", "output": "Workplace betrayal is particularly hurtful because you spend so much time together. This reflects on their character, not yours." },
    { "input": "I shared something personal and it got spread around.", "output": "Having your privacy violated is a serious breach. It's okay to feel angry. You now know who you can confide in." },
    { "input": "My partner was dishonest about something important.", "output": "Dishonesty in a relationship shakes the foundation. Have an honest conversation about what broke and what needs to happen to rebuild." },
    { "input": "Someone I mentored took credit for my work.", "output": "That's incredibly frustrating. Standing up for yourself professionally is important for your career and self-respect." },
    { "input": "I found out my best friend talked behind my back.", "output": "Discovering this hurts, especially from someone close. It might be worth having a real conversation about loyalty and respect." },
    { "input": "My parent broke a promise they made to me.", "output": "Broken promises from parents can affect how you see relationships. Your disappointment is valid, and healing takes patience." },
    { "input": "I was promised a raise but got passed over.", "output": "Broken work promises feel like a personal rejection. Document your contributions and advocate for yourself clearly." },
    { "input": "Someone copied my idea and claimed it as theirs.", "output": "Intellectual theft is frustrating and demoralizing. Stand up for recognition of your work—your contribution matters." },
    { "input": "I caught my partner in a lie they said wasn't important.", "output": "Even small lies can erode trust. Discuss why they felt they needed to hide this from you." },
    { "input": "A friend abandoned me when I needed them most.", "output": "Abandonment when you're vulnerable stings deeply. This shows who your real support system is." },
    { "input": "I was scammed by someone I helped.", "output": "Feeling exploited after you showed kindness is deeply painful. Remember your kindness wasn't wrong—their dishonesty was." },
    { "input": "My boss undermined me in front of the team.", "output": "Public undermining damages your credibility and confidence. Address this privately and professionally." },

    # SUCCESS & ACHIEVEMENT (16-45)
    {
        "input": "I just got promoted at work! I'm so excited!",
        "output": "Congratulations! This is an amazing achievement. Your hard work has paid off. Take a moment to celebrate this success—you deserve it!"
    },
    { "input": "I finally finished my degree after years of studying.", "output": "That's an incredible accomplishment! The dedication and perseverance you showed is remarkable. You should feel immensely proud." },
    { "input": "I got the job I've been dreaming about.", "output": "This is huge! You've reached a major milestone. Take time to celebrate and then embrace this new chapter with confidence." },
    { "input": "I hit my fitness goal and I've never felt better.", "output": "That's fantastic! The discipline and commitment you showed are paying off. Use this momentum to keep going." },
    { "input": "I won an award at work for my contribution.", "output": "Recognition like this validates your hard work. This is a clear sign that your efforts are appreciated and valued." },
    { "input": "I completed a project that I was really proud of.", "output": "Completing something you're proud of is deeply satisfying. That sense of accomplishment is well-deserved." },
    { "input": "I got accepted to my first-choice university.", "output": "Congratulations! This opens so many doors for you. You've earned this opportunity through your hard work." },
    { "input": "I finally published my research paper.", "output": "Getting published is a major achievement in academia. Your work will now contribute to your field. That's incredible!" },
    { "input": "I landed my first client as a freelancer.", "output": "That's a huge step! Building your own business takes courage and skill. Celebrate this win—it's the beginning of something great." },
    { "input": "I learned a new skill that I've been wanting to master.", "output": "Personal growth like this is deeply rewarding. Keep building on this foundation—you're expanding your capabilities." },
    { "input": "I got promoted to team lead and I'm managing people now.", "output": "This is a major career milestone. You've been trusted with leadership—that's a reflection of your competence and potential." },
    { "input": "I finally saved enough money for something I wanted.", "output": "Financial discipline pays off. This accomplishment shows you can set goals and achieve them through dedication." },
    { "input": "I won first place in a competition.", "output": "That's an amazing achievement! You competed and came out on top. Let yourself fully enjoy this victory." },
    { "input": "I got a perfect score on an exam I was nervous about.", "output": "Your preparation and knowledge paid off. This proves you're more capable than you might have doubted." },
    { "input": "I was recognized as employee of the month.", "output": "This formal recognition shows your impact is visible. You're making a real difference in your workplace." },
    { "input": "I completed my first marathon.", "output": "Finishing a marathon is an incredible physical and mental achievement. You've proven to yourself what you're capable of." },
    { "input": "I got a scholarship that will change my future.", "output": "Scholarships recognize both your merit and potential. This is a life-changing opportunity you've earned." },
    { "input": "I successfully negotiated a better salary.", "output": "Advocating for yourself shows confidence and business sense. This increase reflects your true value." },
    { "input": "I delivered a presentation that went really well.", "output": "Successful public speaking builds confidence. Your preparation clearly showed, and people responded well." },
    { "input": "I finally achieved something I thought was impossible.", "output": "Reaching what seemed impossible changes how you see yourself. You're more capable than you realized." },
    { "input": "I got promoted twice in one year!", "output": "This rapid advancement shows your organization values you highly. You're clearly on an excellent career trajectory." },
    { "input": "I successfully launched my own business.", "output": "Entrepreneurship takes incredible courage and vision. You've turned an idea into reality—that's remarkable." },
    { "input": "I won a grant for my research project.", "output": "Grant funding is highly competitive and validates your work's importance. This enables your research to move forward." },
    { "input": "I became fluent in a second language.", "output": "Language mastery is a huge cognitive achievement. The doors this opens for you are now wide." },
    { "input": "I made the professional transition I've always wanted.", "output": "Career transitions take courage. You've bet on yourself and made a bold move toward fulfillment." },
    { "input": "I finally finished my novel after years of writing.", "output": "Completing a creative work of that scale is deeply meaningful. You've created something that will persist." },
    { "input": "I got nominated for an industry award.", "output": "Nominations like this show peer recognition. Your work is being noticed and valued by experts in your field." },
    { "input": "I successfully paid off all my debt.", "output": "Financial freedom is liberating. The discipline this took shows incredible strength of character." },
    { "input": "I achieved the sales target I set for myself.", "output": "Hitting your targets shows goal-setting and execution skills. You're driving your own success." },
    { "input": "I got certified in a professional skill I studied for.", "output": "Certifications validate expertise. You now have official recognition of your competence in this area." },

    # ANXIETY & WORRY (46-75)
    {
        "input": "I'm worried about my exam next week.",
        "output": "It's natural to feel anxious before exams. Channel that energy into preparation. Study smart, get good rest, and remember that you've overcome challenges before."
    },
    { "input": "I have a job interview tomorrow and I'm nervous.", "output": "Interview anxiety is completely normal. Remember they want to find a good fit as much as you do. You have value to offer." },
    { "input": "I'm anxious about what people think of me.", "output": "Worrying about others' judgment is common, but you can't control their thoughts. Focus on being authentic and kind." },
    { "input": "I'm scared about my health test results.", "output": "Medical anxiety is tough. While you wait, try grounding techniques. Whatever the results, you'll face them." },
    { "input": "I'm worried I'm behind in my career compared to peers.", "output": "Comparing timelines causes unnecessary stress. Your path is unique. Progress matters more than pace." },
    { "input": "I'm anxious about starting something new.", "output": "New things bring uncertainty, which naturally triggers anxiety. This feeling usually fades as you become familiar with the new situation." },
    { "input": "I worry constantly about money.", "output": "Financial anxiety is real, but constant worry doesn't help. Focus on actionable steps you can take to improve your situation." },
    { "input": "I'm scared I'll mess up this opportunity.", "output": "Fear of failure before you've even started can be paralyzing. Remember that imperfection is part of learning." },
    { "input": "I'm anxious about my relationship's future.", "output": "Relationship worries often come from caring deeply. Have honest conversations about your concerns rather than letting them build." },
    { "input": "I'm worried about aging and time passing.", "output": "Existential anxiety about time is real. Rather than fearing passage, focus on making moments meaningful." },
    { "input": "I'm scared of failing the training program.", "output": "Training anxiety is about growth. Remember many people feel this way, and most get through it successfully." },
    { "input": "I'm anxious about public speaking at the conference.", "output": "Public speaking fear is one of the most common anxieties. With practice, it becomes more manageable." },
    { "input": "I worry my ideas aren't good enough to share.", "output": "Idea anxiety often stems from perfectionism. Share your thoughts anyway—people benefit from different perspectives." },
    { "input": "I'm scared about moving to a new place.", "output": "Moving anxiety involves fear of the unknown. Build excitement by exploring your new area and connecting with people." },
    { "input": "I'm anxious that I don't fit in at my new job.", "output": "Fitting-in anxiety usually fades as you get to know people. Give yourself time to settle in." },
    { "input": "I worry I'm not smart enough for this role.", "output": "Imposter syndrome makes capable people doubt themselves. Your hiring shows others see your capability." },
    { "input": "I'm scared about commitment in my relationship.", "output": "Commitment anxiety is about fear of loss. It's okay to have concerns—talk them through with your partner." },
    { "input": "I'm anxious about disappointing my parents.", "output": "Parent-anxiety often comes from wanting approval. Your life is yours to live, though open dialogue helps." },
    { "input": "I worry my partner will leave me.", "output": "Abandonment anxiety is rooted in fear. Build security through trust and open communication." },
    { "input": "I'm scared about the side effects of medication.", "output": "Medical anxiety is valid. Talk to your doctor about specific concerns—they can address them directly." },
    { "input": "I'm anxious about presenting my work to leadership.", "output": "Presenting to leadership triggers performance anxiety. Prepare well and remember leaders want to see you succeed." },
    { "input": "I worry constantly about what could go wrong.", "output": "Catastrophic thinking fuels anxiety spirals. When you notice this, gently redirect to what's actually in front of you." },
    { "input": "I'm scared about whether I made the right choice.", "output": "Decision anxiety lingers when choices feel high-stakes. Trust that you made the best choice with the info you had." },
    { "input": "I'm anxious about being alone.", "output": "Loneliness anxiety makes solitude feel unsafe. It may help to reframe alone time as opportunity for self-care." },
    { "input": "I worry about running out of time to achieve my dreams.", "output": "Time pressure anxiety is real in modern life. You have more time than you think—focus on present action." },
    { "input": "I'm scared about financial instability.", "output": "Financial security anxiety is valid given economic uncertainty. Build small safety nets where you can." },
    { "input": "I'm anxious about whether people actually like me.", "output": "Likeability anxiety often comes from needing reassurance. Remember that being yourself is more valuable than being liked by everyone." },
    { "input": "I worry I'm wasting my potential.", "output": "Potential anxiety stems from high expectations. Remember that progress is non-linear and every step counts." },
    { "input": "I'm scared about the unknown future.", "output": "Future uncertainty triggers existential anxiety. You don't control the future—focus on making today meaningful." },
    { "input": "I'm anxious about being judged for my choices.", "output": "Judgment anxiety makes vulnerability scary. Share your authentic self with people you trust." },

    # JOB LOSS & MAJOR SETBACKS (76-105)
    {
        "input": "I lost my job today and don't know what to do.",
        "output": "Job loss is tough, but it's also an opportunity. Take time to grieve the change, then start planning your next steps. You have skills and resilience."
    },
    { "input": "I was laid off along with hundreds of others.", "output": "Mass layoffs feel depersonalizing. Remember this isn't a reflection of your abilities—it's economic restructuring." },
    { "input": "I was fired and feel humiliated.", "output": "Getting fired stings emotionally. Use this as a learning moment and opportunity to reassess what you want." },
    { "input": "My role was eliminated and I'm uncertain about next steps.", "output": "Role elimination is about business needs, not your value. Update your resume and start exploring what's next." },
    { "input": "I didn't get the promotion I was expecting.", "output": "Missed promotions feel like rejection. Ask for feedback on what would help you for next time." },
    { "input": "My business failed and I lost money.", "output": "Business failure is painful but teaches invaluable lessons. Many successful people have failed businesses." },
    { "input": "I was demoted at work and feel ashamed.", "output": "Demotion feels like a step backward. Use it as motivation to reassess your role and plan improvements." },
    { "input": "I missed out on a major opportunity.", "output": "Missed opportunities create regret, but there are always more opportunities ahead." },
    { "input": "I wasn't selected for the project I wanted.", "output": "Project selection disappointments are common. Ask what you can do to be considered next time." },
    { "input": "My contract wasn't renewed and I'm worried about income.", "output": "Contract non-renewal is uncertain. This pushes you to explore new opportunities you might not have otherwise." },
    { "input": "I was rejected from the graduate program I wanted.", "output": "Academic rejection is tough. There are other paths, and rejection doesn't reflect your potential." },
    { "input": "I lost a major client in my business.", "output": "Losing clients hurts revenue and confidence. Use this to improve service or find new clients." },
    { "input": "My internship didn't convert to a full-time offer.", "output": "Internship non-conversions are disappointing. This isn't about your worth—other roles await." },
    { "input": "I was passed over for someone with less experience.", "output": "This feels particularly unfair, but sometimes politics or fit matter more than experience. Advocate for yourself next time." },
    { "input": "I quit my job impulsively and now regret it.", "output": "Impulsive resignations create uncertainty. Now focus on finding something better quickly." },
    { "input": "I failed the professional certification exam.", "output": "Certification failures are discouraging. You now know what to study for next attempt." },
    { "input": "My hours were cut and my income dropped significantly.", "output": "Hour cuts create financial stress. Look for supplementary income or better opportunities." },
    { "input": "I was told my role is being outsourced.", "output": "Outsourcing feels like personal replacement. It's really about cost management. Plan your transition." },
    { "input": "I missed a deadline at work and feel terrible.", "output": "Missed deadlines happen. Learn what contributed and adjust your approach for next time." },
    { "input": "I was skipped over for a performance bonus.", "output": "Bonus denials feel like lack of recognition. Ask specifically what would earn bonus next time." },
    { "input": "My startup funding fell through.", "output": "Funding rejection derails dreams, but setbacks often precede better solutions. Regroup and pivot." },
    { "input": "I was rejected from the team I wanted to join.", "output": "Team rejections feel personal but are often about specific skill gaps. Work on those areas." },
    { "input": "My suggestion was rejected in the meeting.", "output": "Idea rejection feels like personal rejection, but ideas succeed or fail—not people." },
    { "input": "I lost the competition I trained hard for.", "output": "Losing after putting in effort stings. But training made you better regardless of the outcome." },
    { "input": "My proposal wasn't approved by leadership.", "output": "Proposal rejection is about the idea needing refinement, not your competence." },
    { "input": "I found out I'm being investigated at work.", "output": "Investigations are stressful and uncertain. Get clarity on what's happening and what you can do." },
    { "input": "My team project completely failed.", "output": "Project failures are painful learning experiences. Analyze what went wrong and do better next time." },
    { "input": "I was told my performance isn't meeting expectations.", "output": "Performance concerns hurt but are fixable. Work with your manager on a clear improvement plan." },
    { "input": "I lost the client pitch to a competitor.", "output": "Losing pitches stings but teaches you about competition. Improve your pitch for next time." },
    { "input": "My company is going through layoffs and I'm nervous.", "output": "Layoff uncertainty is unsettling. Update your resume now and stay on top of opportunities." },

    # LOVE & RELATIONSHIPS (106-135)
    {
        "input": "My partner said they love me for the first time.",
        "output": "That's such a beautiful moment! This is a meaningful milestone. Cherish this feeling and let yourself be open to this connection."
    },
    { "input": "I just started dating someone and I'm really into them.", "output": "New love feelings are magical. Let yourself enjoy this excitement while keeping realistic expectations." },
    { "input": "My partner proposed and I said yes!", "output": "An engagement is a major life milestone filled with hope and joy. Savor this moment before planning." },
    { "input": "I reconnected with someone from my past and it feels right.", "output": "Reconnection can spark something real. Let it develop naturally and see where it goes." },
    { "input": "My partner showed incredible kindness when I needed support.", "output": "These moments strengthen bonds. Your partner showing up matters. Let yourself feel grateful and supported." },
    { "input": "We had an amazing conversation and felt deeply understood.", "output": "Deep understanding in a relationship is rare and precious. Moments like this are the foundation of lasting connection." },
    { "input": "My partner surprised me with something thoughtful.", "output": "Thoughtful surprises show you're known and cared for. Let yourself feel special and appreciated." },
    { "input": "I told my partner my deepest fear and they responded with love.", "output": "Vulnerability met with acceptance is healing. This kind of support strengthens relationships deeply." },
    { "input": "We laughed so hard together about something silly.", "output": "Laughter together is bonding. These moments of levity strengthen connection." },
    { "input": "My partner supported my dream even when it was risky.", "output": "A partner who believes in you unconditionally is invaluable. This kind of support matters immensely." },
    { "input": "I realized I truly love this person.", "output": "Realizations of love are profound. Whether you say it aloud or not, let yourself feel this deep connection." },
    { "input": "We resolved a major conflict and feel closer than before.", "output": "Conflict resolution strengthens bonds. You've learned to navigate difficulties together." },
    { "input": "My partner makes me want to be a better person.", "output": "Healthy love is inspiring. The fact that they bring out your best self is meaningful." },
    { "input": "I feel safe and secure with my partner.", "output": "Emotional safety is foundational to love. This sense of security is precious." },
    { "input": "We share the same values and life goals.", "output": "Alignment on values is crucial for long-term partnership. This foundation is strong." },
    { "input": "My partner was there for me when nobody else was.", "output": "Partners who show up during tough times reveal their true colors. That loyalty matters." },
    { "input": "We're taking the next step together—moving in!", "output": "Moving in together is an exciting new chapter. Enjoy building your life together." },
    { "input": "I feel attractive and desired by my partner.", "output": "Feeling desired strengthens self-worth. Your partner seeing your beauty is validating." },
    { "input": "We're planning our future together and it excites me.", "output": "Shared future planning is a sign of commitment. Let this excitement fuel your partnership." },
    { "input": "My partner apologized sincerely and we moved forward.", "output": "Genuine apologies show respect. Moving past conflicts builds trust and understanding." },
    { "input": "I met my partner's family and it went really well.", "output": "Family acceptance matters. If it went well, you're becoming part of a larger community." },
    { "input": "My partner believed in me when I doubted myself.", "output": "External belief can reignite your own. A partner's confidence in you is powerful support." },
    { "input": "We had the best date night ever.", "output": "Great date nights remind you why you chose this person. Cherish these moments." },
    { "input": "My partner made a sacrifice for our relationship.", "output": "Sacrifices show prioritization. When someone chooses you, let that sink in." },
    { "input": "I feel like I can be completely myself around my partner.", "output": "Authenticity in relationships is rare and precious. This kind of acceptance is invaluable." },
    { "input": "My partner listened to me without trying to fix everything.", "output": "Being heard without judgment is healing. This kind of presence matters deeply." },
    { "input": "We experienced something beautiful together and bonded over it.", "output": "Shared experiences create memories and deepen connection. Treasure these moments." },
    { "input": "My partner helped me through a difficult time.", "output": "Support during hardship reveals a partner's character. That presence meant everything." },
    { "input": "I feel excited about growing old with this person.", "output": "Long-term partnership excitement shows real commitment and vision. Enjoy that sense of future." },
    { "input": "My partner said they're proud of me.", "output": "Hearing pride from someone you love is deeply affirming. That recognition strengthens you." },

    # SELF-DOUBT & IMPOSTER SYNDROME (136-165)
    { "input": "I feel like a fraud in my position.", "output": "Imposter syndrome affects high achievers. Your presence in this role means you belong here." },
    { "input": "I don't think I'm capable of this project.", "output": "Doubt before challenges is normal. You've overcome difficulties before—you can do this too." },
    { "input": "I'm scared I'm not smart enough for this field.", "output": "Intelligence is multifaceted. The fact that you're here shows you have what it takes." },
    { "input": "Everyone else seems to know what they're doing but I'm lost.", "output": "Most people feel this way—they're just hiding it better. Speak up and ask for help." },
    { "input": "I don't deserve this opportunity.", "output": "Worth and deserving aren't about luck—they're about what you bring. You earned this." },
    { "input": "I'm the least experienced person on my team.", "output": "Being less experienced is temporary. You'll learn and grow. Everyone starts somewhere." },
    { "input": "I feel like I got lucky instead of earning my success.", "output": "Luck plays a part, but your skills created the opportunity. Own your success." },
    { "input": "I'm worried people will discover I'm incompetent.", "output": "Competence anxiety often masks real capability. Your work speaks for itself." },
    { "input": "I don't think I'm creative enough for this role.", "output": "Creativity develops with practice and confidence. Your unique perspective has value." },
    { "input": "I feel like I'm pretending to be confident.", "output": "Many people fake confidence until it becomes real. Keep doing that—it works." },
    { "input": "I don't belong in this room with all these experts.", "output": "You wouldn't be here if you didn't belong. Expertise is a spectrum—you're on it." },
    { "input": "I'm not as talented as my colleagues.", "output": "Comparing talents breeds self-doubt. Everyone has different strengths. Yours are real." },
    { "input": "I feel like I'm not progressing fast enough in my career.", "output": "Progress isn't linear. Your pace is your own. Focus on growth, not speed." },
    { "input": "I wonder if I actually earned my degree or just got lucky.", "output": "You earned it. The work you put in was real. Your education is legitimate." },
    { "input": "I don't feel like I'm adding value at work.", "output": "Not seeing your own value is common for self-doubters. Ask colleagues—they see it." },
    { "input": "I'm scared I'll be exposed as not knowing what I'm doing.", "output": "Many successful people feel this way. Admitting you don't know everything is actually a strength." },
    { "input": "I think I only got this job because of who I know.", "output": "Connections help, but your skills matter too. Both luck and merit got you here." },
    { "input": "I'm not as good at my job as people think I am.", "output": "Self-assessment is often harsher than reality. Your results show your capability." },
    { "input": "I feel like a fake when people praise my work.", "output": "Praise makes self-doubters uncomfortable. But it's based on real results—accept it." },
    { "input": "I don't think I can handle more responsibility.", "output": "Growth means doing things you're unsure about. You're more capable than you think." },
    { "input": "I'm worried I'll embarrass myself in this meeting.", "output": "Vulnerability sometimes happens, and that's human. Most people are focusing on themselves, not judging you." },
    { "input": "I feel like I'm failing even though my numbers are good.", "output": "Self-doubt doesn't respond to logic, but it does respond to time. Let yourself feel good about results." },
    { "input": "I don't think I'm good enough to teach others.", "output": "You don't need to be perfect to teach. Your experience and perspective have value." },
    { "input": "I feel unqualified for leadership.", "output": "Leaders doubted themselves too. Your role exists because the organization believes you're capable." },
    { "input": "I'm worried I don't have the right background for this.", "output": "Non-traditional backgrounds bring fresh perspectives. Your unique path is an asset." },
    { "input": "I feel like everyone else belongs here more than I do.", "output": "Belonging isn't determined by perfection. You're here because you matter." },
    { "input": "I don't think I'm creative or innovative enough.", "output": "Creativity comes from showing up and trying. Your attempts matter even when imperfect." },
    { "input": "I'm scared of asking questions because it makes me look stupid.", "output": "Curious people ask questions. That's a sign of intelligence, not stupidity." },
    { "input": "I feel like a burden to my team.", "output": "Teams exist because people need each other. You contribute more than you realize." },
    { "input": "I wonder if I'll ever feel confident in my abilities.", "output": "Confidence builds gradually. With each success, doubt weakens. You're on the right path." },

    # OVERWHELM & STRESS (166-195)
    { "input": "I feel completely overwhelmed by my responsibilities.", "output": "That sounds exhausting. When everything piles up, it's okay to slow down and take things one step at a time. You don't have to handle everything at once." },
    { "input": "I have too much on my plate and don't know where to start.", "output": "Overwhelm often comes from seeing the whole picture. Break it down into smaller tasks and tackle one at a time." },
    { "input": "I feel stretched too thin between work and family.", "output": "Juggling multiple areas is exhausting. Set boundaries and accept that you can't do everything perfectly." },
    { "input": "Everything feels urgent and I can't keep up.", "output": "When everything feels urgent, nothing is. Prioritize ruthlessly and let go of less important things." },
    { "input": "I'm drowning in work deadlines.", "output": "Deadline stress is real but manageable. Communicate about timelines and ask for help if needed." },
    { "input": "I have so many projects and don't know which to focus on.", "output": "Project overload paralyzes progress. Pick the most important one and ignore the rest for now." },
    { "input": "I feel like I'm barely holding it together.", "output": "Feeling like you're barely managing is a sign you need to cut something out. What's least essential?" },
    { "input": "Everyone needs something from me and I'm exhausted.", "output": "Setting boundaries isn't selfish—it's necessary. You can't pour from an empty cup." },
    { "input": "I have too many commitments and can't keep my promises.", "output": "Over-committing creates stress and disappointment. It's okay to say no to some things." },
    { "input": "I'm stressed about money and work at the same time.", "output": "Dual stress compounds. Address the most urgent one first, then make a plan for the other." },
    { "input": "I feel like my life is out of control.", "output": "Loss of control often comes from taking on too much. Focus on what you can actually control." },
    { "input": "I'm overwhelmed by decisions I need to make.", "output": "Decision fatigue is real. Make important decisions early in the day when you have mental energy." },
    { "input": "I have back-to-back meetings and no time to think.", "output": "Constant meetings are cognitively exhausting. Block time for focus and protect it." },
    { "input": "I feel like I'm always behind on something.", "output": "Chronic behind-ness means your expectations need adjustment. What can actually wait?" },
    { "input": "I'm juggling so many things that nothing gets my full attention.", "output": "Multitasking reduces quality on everything. Focus on one thing at a time, even if brief." },
    { "input": "I feel pressure to be perfect at everything.", "output": "Perfectionism creates burnout. Done is better than perfect in most things." },
    { "input": "I'm stressed about my finances, health, and work all at once.", "output": "Triple stress is overwhelming. Tackle one area this week, then move to the next." },
    { "input": "I don't have time to take care of myself anymore.", "output": "Self-care isn't selfish—it's essential. Even small things help: a short walk, good sleep." },
    { "input": "I feel like I'm running on empty.", "output": "Running on empty leads to burnout. You need rest, not more caffeine. Take time to recover." },
    { "input": "Everything feels urgent and I can't relax.", "output": "Chronic urgency is stressful. Practice saying 'this can wait' and actually believing it." },
    { "input": "I'm overwhelmed by how much I need to learn.", "output": "Learning curves are steep but manageable. You don't need to master everything at once." },
    { "input": "I have too many goals and feel scattered.", "output": "Too many goals dilutes effort. Pick three priorities and focus there." },
    { "input": "I'm stressed about my commute and work hours.", "output": "Commute stress compounds work stress. Explore options: different timing, working from home, or closer work." },
    { "input": "I feel like I'm never going to catch up.", "output": "Catch-up mindset is exhausting. Accept current workload and work from there." },
    { "input": "I'm overwhelmed by my inbox and notifications.", "output": "Information overload adds stress. Turn off non-essential notifications and batch email checking." },
    { "input": "I have too much to do and not enough time.", "output": "This is almost everyone's reality. Something has to give. What's actually optional?" },
    { "input": "I feel pressure from everyone expecting things from me.", "output": "Everyone's expectations aren't your responsibility. You get to choose what you take on." },
    { "input": "I'm stressed about my performance reviews.", "output": "Review anxiety is common. Document your achievements and prepare to speak about your value." },
    { "input": "I feel like I'm always working, even on weekends.", "output": "Work bleeding into personal time creates burnout. Set strict boundaries on when work ends." },
    { "input": "I'm overwhelmed and don't know where to start.", "output": "Overwhelm paralyzes. Pick the smallest thing you can do right now and just do that." },

    # GRIEF & LOSS (196-215)
    { "input": "I lost someone I love and can't process it.", "output": "Grief is complex and non-linear. Let yourself feel whatever comes without judgment." },
    { "input": "My best friend moved away and I miss them.", "output": "Distance changes friendships but doesn't end them. Stay connected in whatever ways you can." },
    { "input": "I'm grieving a relationship that ended.", "output": "Relationship grief is real even if it's healthy it ended. Mourn what you shared, then move forward." },
    { "input": "Someone important passed away and I don't know how to move forward.", "output": "Death creates a void. Lean on community, allow time, and honor their memory however feels right." },
    { "input": "I lost my job and feel like I lost my identity.", "output": "Jobs feel like identity, but they're not. You're more than your role. Rediscover yourself." },
    { "input": "I'm dealing with the loss of my childhood home.", "output": "Losing places is grief too. The memories stay with you even as physical spaces change." },
    { "input": "My pet died and I can't stop crying.", "output": "Pet loss is legitimate grief. That bond was real, and mourning it is appropriate." },
    { "input": "I lost a friendship I thought would last forever.", "output": "Friendship endings can be as painful as romantic ones. Grieve the loss and remember the good times." },
    { "input": "I'm grieving the future I imagined that won't happen.", "output": "Grief for futures not lived is valid. Let go of what won't be and find new possibilities." },
    { "input": "Someone I care about is terminally ill and I'm grieving early.", "output": "Anticipatory grief is real. Spend quality time now and express what matters." },
    { "input": "I lost my health and feel like I lost myself.", "output": "Health loss changes life. Grieve what you've lost while discovering new capabilities." },
    { "input": "I'm dealing with multiple losses at once.", "output": "Compound grief is overwhelming. Get support. You don't process multiple losses alone well." },
    { "input": "I feel guilty that I'm not grieving 'correctly' for this loss.", "output": "There's no correct way to grieve. Your way is valid even if it looks different from others'." },
    { "input": "I've lost hope for something I wanted deeply.", "output": "Hope loss feels like death. Sit with that, then look for new things to hope for." },
    { "input": "I'm grieving the loss of who I used to be.", "output": "Identity shifts are losses. Acknowledge what changed, then explore your new self." },
    { "input": "Someone died and I didn't get to say goodbye.", "output": "Not getting closure hurts. Write a letter, visit their grave, or honor them in some way." },
    { "input": "I lost my career due to circumstances beyond my control.", "output": "Career loss through no fault of yours is unfair. Grieve it, then rebuild." },
    { "input": "I'm dealing with the loss of innocence in my life.", "output": "Growing up always involves innocence loss. You're wiser now, even if it hurts." },
    { "input": "Someone I depended on is no longer in my life.", "output": "Losing support systems is destabilizing. Find new support as you process this loss." },
    { "input": "I'm grieving my youth and time passing.", "output": "Aging brings loss of youth. Find meaning in your current stage rather than mourning what's past." },

    # SHAME & GUILT (216-245)
    { "input": "I feel ashamed of something I did.", "output": "Shame is painful but also human. Acknowledge it, make amends if possible, and learn from it." },
    { "input": "I did something I feel guilty about.", "output": "Guilt is a sign you care about your values. Use it to correct course and do better." },
    { "input": "I feel ashamed of my family situation.", "output": "Family circumstances don't define you. You're not responsible for others' choices." },
    { "input": "I'm ashamed of how I've treated someone I care about.", "output": "That guilt shows you care. Apologize sincerely and commit to better behavior." },
    { "input": "I feel guilty for not being there for my friend.", "output": "You can't be everything to everyone. Do what you can now to repair the relationship." },
    { "input": "I'm ashamed of my body.", "output": "Body shame is cultural, not truth. Your body is worthy of care and respect as it is." },
    { "input": "I feel guilty about my privilege.", "output": "Guilt about advantage is natural. Channel it into action and use privilege responsibly." },
    { "input": "I'm ashamed of my past decisions.", "output": "Past selves make the decisions they know how to make. Forgive yourself and do better now." },
    { "input": "I feel guilty for having needs.", "output": "Needs aren't selfish. Everyone has them. Meeting your own needs isn't shame-worthy." },
    { "input": "I'm ashamed I didn't stand up when I should have.", "output": "Missed opportunities to stand up hurt. Use this to know yourself better and be braver next time." },
    { "input": "I feel guilty for not helping when I could have.", "output": "That guilt can push you to help now. Use it as motivation for future action." },
    { "input": "I'm ashamed of how anxious I get.", "output": "Anxiety is involuntary. There's no shame in your body's stress response." },
    { "input": "I feel guilty for enjoying myself.", "output": "Joy isn't something to feel guilty about. You're allowed to experience happiness." },
    { "input": "I'm ashamed of my mistakes at work.", "output": "Work mistakes happen to everyone. What matters is how you handle them now." },
    { "input": "I feel guilty for not visiting my family enough.", "output": "Guilt about family time is common. Do what you can with the bandwidth you have." },
    { "input": "I'm ashamed I had to ask for financial help.", "output": "Needing help doesn't shame you. It shows wisdom to know when to ask." },
    { "input": "I feel guilty for my partner taking care of me.", "output": "Partnership includes being cared for. Let yourself receive without guilt." },
    { "input": "I'm ashamed of my financial situation.", "output": "Finances are circumstantial. You're not defined by money. Focus on what you can control." },
    { "input": "I feel guilty for working while I'm a parent.", "output": "Working parents aren't neglectful—you're providing and modeling work ethic." },
    { "input": "I'm ashamed I cried in front of others.", "output": "Tears are normal. Letting others see your emotions shows strength, not weakness." },
    { "input": "I feel guilty for setting a boundary.", "output": "Healthy boundaries aren't selfish. They're necessary for your wellbeing." },
    { "input": "I'm ashamed of how much I earned/spent.", "output": "Financial choices are personal. As long as they work for you, there's no shame." },
    { "input": "I feel guilty for enjoying a day off while others work.", "output": "You deserve rest. Rest isn't something to feel guilty about." },
    { "input": "I'm ashamed of my health habits.", "output": "Body shame about habits can be motivating, but not punitive. Make small changes, be kind to yourself." },
    { "input": "I feel guilty for not being perfect.", "output": "Nobody's perfect. Accepting your humanness is maturity, not failure." },
    { "input": "I'm ashamed I got angry with someone.", "output": "Anger is valid. How you express it matters. If you acted poorly, apologize and adjust." },
    { "input": "I feel guilty for my success compared to others.", "output": "Your success doesn't diminish others. Celebrate it while lifting others up." },
    { "input": "I'm ashamed of my appearance.", "output": "Appearance insecurity is common but not truth. You're worthy regardless of looks." },
    { "input": "I feel guilty for time I spent on myself.", "output": "Self-care time isn't selfish. It's necessary for being present for others." },
    { "input": "I'm ashamed of how I handled a conflict.", "output": "Conflict mishandling is a learning moment. Apologize, reflect, and do better next time." },

    # HAPPINESS & CONTENTMENT (246-265)
    { "input": "I'm really happy right now and life feels good.", "output": "This is wonderful. These moments of peace and happiness are precious. Savor them." },
    { "input": "I woke up today feeling grateful for my life.", "output": "Gratitude is a powerful emotion. Let yourself appreciate the good things around you." },
    { "input": "I'm content with where I am, even if it's not perfect.", "output": "Contentment without complacency is wisdom. You can be happy now while still growing." },
    { "input": "I finally feel at peace with my past.", "output": "Peace with your history is freedom. That's a significant emotional milestone." },
    { "input": "I'm happy with my life choices and direction.", "output": "Confidence in your path is affirming. Trust in your direction and keep moving forward." },
    { "input": "I feel deeply satisfied with my work.", "output": "Work satisfaction is rare and precious. You've found something meaningful." },
    { "input": "I'm in a really good place mentally and emotionally.", "output": "Mental wellbeing is a gift. Protect it by maintaining what's working." },
    { "input": "I feel genuine joy when I think about my future.", "output": "Optimistic joy about the future is powerful. That hope will fuel your actions." },
    { "input": "I'm happy with who I've become.", "output": "Self-acceptance is beautiful. You're living authentically and that matters." },
    { "input": "I feel blessed by my relationships.", "output": "Appreciation for good relationships amplifies their value. Express gratitude to those people." },
    { "input": "I'm in a really good mood today.", "output": "Good moods are to be enjoyed. Let yourself sit in this positive feeling." },
    { "input": "I feel truly comfortable being myself around my people.", "output": "Authentic belonging is one of life's greatest gifts. Treasure these people." },
    { "input": "I'm happy with my personal growth this year.", "output": "Growth recognition is important. You're becoming the person you want to be." },
    { "input": "I feel at peace with my body and health.", "output": "Body peace is rare and valuable. You're treating yourself with kindness." },
    { "input": "I'm genuinely excited about my future plans.", "output": "Excitement about what's coming is energizing. Let that propel you forward." },
    { "input": "I feel whole and complete as I am.", "output": "Wholeness doesn't require fixing. This self-acceptance is beautiful." },
    { "input": "I'm happy with my financial situation.", "output": "Financial peace brings freedom. You've built something stable." },
    { "input": "I feel truly loved and appreciated.", "output": "Being loved and knowing it is fundamental. Let yourself fully receive that." },
    { "input": "I'm content with my simple life.", "output": "Simplicity can be deeply satisfying. You've found what matters." },
    { "input": "I feel hopeful and energized about life.", "output": "Hope and energy are the fuel for great things. Use this momentum wisely." },

    # IDENTITY & BELONGING (266-295)
    { "input": "I'm figuring out who I am as a person.", "output": "Identity discovery is ongoing. Give yourself permission to evolve and change." },
    { "input": "I feel like I belong here for the first time.", "output": "Finding belonging is transformative. Treasure communities where you truly fit." },
    { "input": "I'm exploring my cultural identity and it feels empowering.", "output": "Cultural exploration is healing. Connecting with your roots strengthens your sense of self." },
    { "input": "I'm coming out and I'm scared but also relieved.", "output": "Authenticity takes courage. What you're doing matters, and your truth deserves to be known." },
    { "input": "I'm realizing I've been living for others, not myself.", "output": "That realization is painful but freeing. Start making choices for yourself now." },
    { "input": "I finally understand my values and what matters to me.", "output": "Values clarity is powerful. Use this to make decisions aligned with who you are." },
    { "input": "I'm embracing parts of myself I used to hide.", "output": "Self-acceptance is liberating. The world needs your full, authentic self." },
    { "input": "I feel like I've found my tribe.", "output": "Finding people who get you is rare and wonderful. These connections fuel happiness." },
    { "input": "I'm rebuilding my identity after a major life change.", "output": "Identity rebuilding is challenging but creates opportunity for growth. Explore who you want to become." },
    { "input": "I'm confident in who I am and what I believe.", "output": "Confidence in your identity is grounding. Let it guide your choices." },
    { "input": "I'm learning to love the parts of myself I dislike.", "output": "Self-compassion toward parts you dislike is growth. We all have flaws—that's being human." },
    { "input": "I'm standing up for who I am even when others don't understand.", "output": "Authenticity sometimes means standing alone. Your integrity matters more than understanding." },
    { "input": "I'm discovering talents and strengths I didn't know I had.", "output": "Self-discovery of strengths is empowering. You're fuller than you realized." },
    { "input": "I feel like I've found my purpose.", "output": "Purpose discovery is profoundly meaningful. Let this guide your path forward." },
    { "input": "I'm comfortable with being different.", "output": "Embracing difference is strength. The world needs diverse perspectives." },
    { "input": "I'm no longer trying to be who others expect me to be.", "output": "Releasing others' expectations frees you. Live for yourself now." },
    { "input": "I'm becoming who I always wanted to be.", "output": "This journey is beautiful. Keep moving toward your authentic self." },
    { "input": "I'm proud of my heritage and background.", "output": "Heritage pride is affirming. Your background is part of what makes you valuable." },
    { "input": "I'm learning to accept my identity without shame.", "output": "Identity acceptance is liberation. You're worthy as you are." },
    { "input": "I feel like I finally know where I belong in the world.", "output": "Belonging clarity is grounding. You've found your place." },
    { "input": "I'm honoring my true self even when it's challenging.", "output": "Living authentically despite challenges shows character. Keep being you." },
    { "input": "I'm grateful for how different I am from others.", "output": "Celebrating difference is rare. Your uniqueness is your superpower." },
    { "input": "I'm finding my voice and speaking my truth.", "output": "Voice discovery is empowering. Your perspective deserves to be heard." },
    { "input": "I'm comfortable taking up space as I am.", "output": "You deserve to take up space. Stop apologizing for existing." },
    { "input": "I'm building a life that reflects my true values.", "output": "Values-aligned living is fulfilling. Keep building authentically." },
    { "input": "I'm embracing my sensitivity as a strength.", "output": "Sensitivity is often a strength, not weakness. Use it to connect and create." },
    { "input": "I'm learning that I'm enough just as I am.", "output": "The 'enoughness' you seek is already here. Stop looking outside yourself." },
    { "input": "I'm proud of my choices and the person I'm becoming.", "output": "Self-pride in your growth is beautiful. You're on a good path." },
    { "input": "I'm finally letting myself want what I want.", "output": "Want validation is necessary. Your desires are valid—honor them." },
    { "input": "I'm building the life I actually want, not should want.", "output": "This distinction changes everything. Keep choosing what you actually want." },

    # GROWTH & LEARNING (296-345)
    { "input": "I'm learning from my mistakes and becoming better.", "output": "Mistakes are your best teachers. You're using them wisely to grow." },
    { "input": "I tried something new today and it was scary but good.", "output": "Stepping outside comfort zones is how growth happens. You're becoming more resilient." },
    { "input": "I'm reading more and expanding my mind.", "output": "Learning is a lifelong gift. You're investing in yourself—that matters." },
    { "input": "I took a course and gained new skills.", "output": "Investing in education pays dividends. You're building your capabilities intentionally." },
    { "input": "I'm challenging my old beliefs and growing.", "output": "Belief evolution shows intellectual honesty. You're becoming more nuanced and wise." },
    { "input": "I faced a fear and came out stronger.", "output": "Conquering fears is empowering. Each one you face increases your courage." },
    { "input": "I'm developing emotional intelligence.", "output": "Emotional awareness makes you a better person and partner. This growth matters." },
    { "input": "I learned something about myself today.", "output": "Self-discovery is ongoing and valuable. These insights guide better decisions." },
    { "input": "I'm becoming more patient with myself and others.", "output": "Patience is a practice. The fact you're working on it shows growth." },
    { "input": "I'm learning to communicate better in relationships.", "output": "Communication skills transform connections. You're building stronger bonds." },
    { "input": "I'm pushing myself to be more disciplined.", "output": "Discipline is freedom. The constraints you embrace create real options." },
    { "input": "I tried public speaking and survived it.", "output": "Each speaking attempt builds confidence. You're getting better at visibility." },
    { "input": "I'm learning to listen without jumping to advice.", "output": "Deep listening is a gift to others. You're becoming a better friend." },
    { "input": "I'm developing my leadership style.", "output": "Leadership is learned and practiced. You're discovering your authentic way." },
    { "input": "I'm becoming more comfortable with vulnerability.", "output": "Vulnerability takes courage. You're building real connections through honesty." },
    { "input": "I learned a difficult lesson and it shaped me.", "output": "Hard lessons stick and shape wisdom. You're stronger for going through it." },
    { "input": "I'm improving my time management skills.", "output": "Time management is freedom from overwhelm. You're taking control back." },
    { "input": "I'm learning to be more assertive.", "output": "Assertiveness is healthy and necessary. Your needs matter too." },
    { "input": "I'm studying something fascinating and it energizes me.", "output": "Learning that lights you up is precious. Follow these interests—they reveal your purpose." },
    { "input": "I'm becoming more resilient through challenges.", "output": "Resilience is built one challenge at a time. You're stronger than you were." },
    { "input": "I'm learning to accept my body as it changes.", "output": "Body acceptance is peace. You're honoring your vessel through all changes." },
    { "input": "I'm improving my financial literacy.", "output": "Money knowledge is power. You're taking control of your economic life." },
    { "input": "I'm learning to say no without guilt.", "output": "Guilt-free no is a superpower. You're honoring your boundaries." },
    { "input": "I'm becoming more mindful in my daily life.", "output": "Mindfulness transforms everyday moments into presence. You're living more awake." },
    { "input": "I learned how to handle conflict better.", "output": "Conflict skills are rare and valuable. You're becoming better at hard conversations." },
    { "input": "I'm learning to trust my intuition.", "output": "Intuition becomes clearer with practice. Your gut wisdom is worth listening to." },
    { "input": "I'm developing a meditation practice.", "output": "Meditation is training for your mind. You're building calm from within." },
    { "input": "I'm learning about my attachment style and it's helping.", "output": "Self-understanding transforms relationships. You're becoming more conscious." },
    { "input": "I'm improving my critical thinking skills.", "output": "Critical thinking prevents manipulation. You're becoming wiser in your choices." },
    { "input": "I'm learning to be more authentic in social situations.", "output": "Authentic socializing is exhausting less. You're finding your people." },

    # RELATIONSHIPS & CONNECTION (346-395)
    { "input": "My friend surprised me with their kindness today.", "output": "Good friends show up in small ways too. These moments strengthen bonds." },
    { "input": "I'm deepening my friendships through vulnerability.", "output": "Real friendships deepen through honesty. You're building genuine connections." },
    { "input": "I had a great conversation with someone I admire.", "output": "Learning from people you respect accelerates growth. That exchange was valuable." },
    { "input": "My family supported me in a way I really needed.", "output": "Family support when you need it most matters deeply. Let that sink in." },
    { "input": "I reconnected with an old friend and it felt natural.", "output": "Rekindled friendships often have special meaning. Treasure reconnections like these." },
    { "input": "I set healthy boundaries with my family.", "output": "Family boundaries are necessary for health. You're protecting yourself wisely." },
    { "input": "My team came together to solve a problem.", "output": "Teamwork makes the dream work. You're building with people who care." },
    { "input": "I'm building community around shared interests.", "output": "Communities built on passion are energizing. You're finding your people." },
    { "input": "I had a conflict with a friend but we worked through it.", "output": "Friendships that survive conflict are stronger. You're learning what real friendship looks like." },
    { "input": "Someone asked for my help and I was there for them.", "output": "Being needed is affirming. Your support matters to that person." },
    { "input": "I'm mentoring someone and watching them grow.", "output": "Mentoring is fulfilling. You're passing on wisdom and helping someone rise." },
    { "input": "My partner and I laughed until we cried together.", "output": "Laughter with someone you love is bonding. These moments are life's treasure." },
    { "input": "I met someone new and we have great chemistry.", "output": "New connections with good energy are exciting. See where this grows." },
    { "input": "I'm part of a group that truly gets me.", "output": "Finding your people changes everything. You're no longer alone in your experience." },
    { "input": "My colleague became my friend.", "output": "Work friendships can be deep. You're building connection in all areas of life." },
    { "input": "I'm more honest with my family than I've ever been.", "output": "Family honesty builds real intimacy. You're creating authentic connections." },
    { "input": "I reached out when I was struggling and people showed up.", "output": "Being met when vulnerable is healing. You have more support than you knew." },
    { "input": "I'm teaching someone something they appreciate.", "output": "Being valued for what you know is affirming. Your knowledge matters." },
    { "input": "My inner circle expanded and I'm grateful.", "output": "Close circles that grow intentionally are treasures. You're building real family." },
    { "input": "I'm learning how to be a better friend.", "output": "Friendship is a skill. You're practicing presence and showing up." },
    { "input": "Someone checked in on me when I was quiet.", "output": "Being noticed when you're struggling shows you matter. That care is real." },
    { "input": "I'm in a group chat where I truly belong.", "output": "Communities online or offline that feel safe are precious. You've found your space." },
    { "input": "My partner knows me better than I know myself sometimes.", "output": "Being truly known is rare and beautiful. Cherish someone who sees all of you." },
    { "input": "I made plans with friends and I'm looking forward to it.", "output": "Anticipation of good time together is part of the gift. Enjoy the planning too." },
    { "input": "I'm building deeper connections with my siblings.", "output": "Adult sibling relationships can be profound. You're choosing connection intentionally." },
    { "input": "Someone I admire asked my opinion.", "output": "Your perspective has value. Being asked for it shows they respect you." },
    { "input": "I'm spending quality time with someone I love.", "output": "Intentional time together strengthens bonds. These moments are what matter most." },
    { "input": "My network is supporting my professional growth.", "output": "Good networks lift you up. You're surrounded by people invested in your success." },
    { "input": "I'm being a good influence on people I care about.", "output": "Your example matters. You're modeling good behavior for others." },
    { "input": "I feel seen and understood by my partner.", "output": "Being truly seen in relationship is everything. That understanding is precious." },

    # PERSONAL HEALTH & WELLBEING (396-445)
    { "input": "I'm sleeping better and feeling more rested.", "output": "Quality sleep changes everything. Your body is recovering and strengthening." },
    { "input": "I started exercising and I feel more energy.", "output": "Movement is medicine. You're building strength and boosting mood." },
    { "input": "I'm eating healthier and my body feels better.", "output": "Nutrition choices affect everything. You're honoring your body's needs." },
    { "input": "I finally addressed a health issue I was ignoring.", "output": "Facing health problems head-on is brave. You're taking care of yourself." },
    { "input": "I got my mental health support and it's helping.", "output": "Therapy or counseling is strength. You're investing in your mind." },
    { "input": "I'm more in tune with my body's signals.", "output": "Body awareness is wisdom. You're listening to yourself." },
    { "input": "I'm taking medication and feeling stable.", "output": "Medication when needed is care, not weakness. You're managing your health." },
    { "input": "I quit something that was harming me.", "output": "Protecting your health sometimes means loss. But the gain is worth it." },
    { "input": "I'm drinking more water and taking better care of myself.", "output": "Small habits compound. These choices add up to big health improvements." },
    { "input": "I finally got rest after pushing hard for months.", "output": "Rest is productive. Your body and mind needed this recovery." },
    { "input": "I'm managing my stress better with techniques I learned.", "output": "Stress management tools work when you use them. You're taking control." },
    { "input": "I'm more aware of my mental health triggers.", "output": "Awareness is the first step to management. You're being strategic about your triggers." },
    { "input": "I'm establishing a morning routine that energizes me.", "output": "Mornings set the tone. You're building a foundation for good days." },
    { "input": "I went to my medical checkup and got good news.", "output": "Health security is a gift. Take care to maintain this." },
    { "input": "I'm reducing my caffeine and sleeping better.", "output": "Small adjustments compound. You're listening to what your body needs." },
    { "input": "I finally did the dental work I was avoiding.", "output": "Facing health maintenance is responsible self-care. You're protecting yourself." },
    { "input": "I'm more active and my mood has improved.", "output": "Movement therapy is real. Your mood is benefiting from physical activity." },
    { "input": "I'm setting better sleep boundaries with technology.", "output": "Sleep hygiene is important. You're protecting your rest." },
    { "input": "I got a massage and my body feels less tense.", "output": "Physical care releases tension. Your body needed that release." },
    { "input": "I'm working with a therapist on my patterns.", "output": "Therapy is deep work. You're understanding yourself at a new level." },
    { "input": "I'm more mindful about my mental health daily.", "output": "Daily mental health attention prevents crises. You're being proactive." },
    { "input": "I started yoga and I feel more centered.", "output": "Yoga integrates body and mind. You're building centeredness." },
    { "input": "I'm limiting alcohol and feeling clearer.", "output": "Reducing substances clears your mind. You're thinking more clearly." },
    { "input": "I got vaccinated and I feel protected.", "output": "Taking health precautions is wise. You're protecting yourself." },
    { "input": "I'm eating mindfully instead of emotionally.", "output": "Conscious eating changes your relationship with food. You're healing your patterns." },
    { "input": "I started tracking my mental health and see patterns.", "output": "Awareness through tracking helps. You're becoming scientific about your mental health." },
    { "input": "I'm taking time for hobbies that fill my cup.", "output": "Hobbies are necessary, not luxury. You're feeding your soul." },
    { "input": "I finally feel like I'm taking my health seriously.", "output": "Health commitment is self-love. You're prioritizing what matters." },
    { "input": "I'm balancing work and rest much better now.", "output": "Work-life balance is achievable. You're protecting your wellbeing." },
    { "input": "I'm managing my chronic condition better.", "output": "Living well with illness is possible. You're adapting and thriving." },

    # CHALLENGES & OVERCOMING (446-495)
    { "input": "I finally spoke up about something that bothered me.", "output": "Speaking up takes courage. You're honoring your feelings and boundaries." },
    { "input": "I walked away from something toxic.", "output": "Removing toxins from your life is brave. You're choosing health." },
    { "input": "I apologized sincerely and felt relieved.", "output": "Real apologies heal. You're taking responsibility and mending bonds." },
    { "input": "I confronted someone and held my ground.", "output": "Standing firm in confrontation is empowering. You're not backing down." },
    { "input": "I finally let go of something I was holding onto.", "output": "Release is freedom. You're unburdening yourself." },
    { "input": "I pushed through discomfort and succeeded.", "output": "Discomfort is the price of growth. You paid it and won." },
    { "input": "I admitted something hard and found acceptance.", "output": "Honesty followed by acceptance is liberating. You're being seen fully." },
    { "input": "I chose myself for the first time.", "output": "Self-prioritization is healthy. You're learning what matters most." },
    { "input": "I faced my biggest fear and survived.", "output": "Post-fear clarity is remarkable. You're braver than you knew." },
    { "input": "I recovered from a major setback faster this time.", "output": "Your resilience is growing. Bounce-back time is decreasing." },
    { "input": "I stopped making excuses and took action.", "output": "Accountability is powerful. You're owning your choices." },
    { "input": "I asked for help when I would normally suffer alone.", "output": "Help-seeking is strength. You're ending the lone-wolf pattern." },
    { "input": "I changed my mind about something important.", "output": "Changing your mind shows growth. You're willing to evolve." },
    { "input": "I stood up for someone else despite risk.", "output": "Courage for others shows your values. You're a good person." },
    { "input": "I finished something difficult I wanted to quit.", "output": "Perseverance through difficulty is admirable. You're building discipline." },
    { "input": "I examined my biases and changed my behavior.", "output": "Bias examination is humbling and important. You're becoming better." },
    { "input": "I forgave someone who hurt me.", "output": "Forgiveness is freedom from the past. You're releasing resentment." },
    { "input": "I set a boundary that felt impossible to set.", "output": "Impossible boundaries once set feel necessary. You're protecting yourself." },
    { "input": "I spoke truth even when it was unpopular.", "output": "Speaking truth costs sometimes. Your integrity is worth the price." },
    { "input": "I accepted responsibility for my part.", "output": "Owning your role in problems is maturity. You're taking control back." },
    { "input": "I quit something I thought I had to do.", "output": "Releasing shoulds is freeing. You're honoring your real priorities." },
    { "input": "I faced criticism and learned from it.", "output": "Using criticism as data rather than judgment shows wisdom. You're growing." },
    { "input": "I chose integrity over convenience.", "output": "Character is built by hard choices. You chose right." },
    { "input": "I ended a relationship that wasn't working.", "output": "Ending relationships takes courage. You're choosing your wellbeing." },
    { "input": "I admitted I was wrong and felt lighter.", "output": "Admitting error is freedom. Wrongness is part of being human." },
    { "input": "I stopped trying to control what I can't control.", "output": "Control release is peace. You're accepting reality." },
    { "input": "I took the road less traveled.", "output": "Diverging from the script takes courage. Your path is valid." },
    { "input": "I faced my shame and found out I survived.", "output": "Facing shame dissipates it. You're stronger than your shame." },
    { "input": "I made a hard decision and I'm at peace with it.", "output": "Hard decisions followed by peace mean you chose right. Trust yourself." },
    { "input": "I started over and it's scary but right.", "output": "Starting over takes everything you have. You've got this." },

    # EVERYDAY MOMENTS & JOY (496-545)
    { "input": "I got to sleep in and it was wonderful.", "output": "Small comforts matter. Let yourself enjoy simple pleasures." },
    { "input": "The weather is beautiful and I'm enjoying it.", "output": "Weather joy is simple and pure. Be present in this moment." },
    { "input": "I made someone laugh and it felt great.", "output": "Making others laugh spreads joy. Your humor matters." },
    { "input": "I had a perfect cup of coffee this morning.", "output": "Small pleasures compound into contentment. Savor these moments." },
    { "input": "I watched a beautiful sunset today.", "output": "Nature's beauty heals. You noticed and that's mindfulness." },
    { "input": "I found something I lost and I'm relieved.", "output": "Recovery of lost things feels like luck. Enjoy the relief." },
    { "input": "I had an unexpected good interaction today.", "output": "Surprises of kindness brighten days. Keep noticing these." },
    { "input": "I laughed at something silly and my day got better.", "output": "Silly laughter is medicine. More of this, please." },
    { "input": "I got a compliment that meant a lot.", "output": "Genuine compliments fill cups. Let yourself hear it." },
    { "input": "I treated myself to something small I wanted.", "output": "Small treats are self-care. You deserve these moments." },
    { "input": "I made someone's day better and felt fulfilled.", "output": "Small acts of kindness ripple. You changed someone's day." },
    { "input": "I had no obligations today and it was blissful.", "output": "Free time is luxury. Enjoy unscheduled moments." },
    { "input": "I listened to a song that lifted my mood.", "output": "Music is mood medicine. Keep your playlist ready." },
    { "input": "I saw something beautiful and stopped to admire it.", "output": "Appreciating beauty requires attention. You're mindful." },
    { "input": "I had a good hair day and I felt more confident.", "output": "Small appearance confidence boosts matter. Enjoy the good-hair-day energy." },
    { "input": "I received a text from someone thinking of me.", "output": "Being thought of is affirming. Someone cares about you." },
    { "input": "I solved something I thought was unsolvable.", "output": "Solutions surprise us when we stop forcing. You figured it out." },
    { "input": "I felt inspired while doing something creative.", "output": "Inspiration flow is life. Keep creating—it matters." },
    { "input": "I had a spontaneous adventure and it was fun.", "output": "Spontaneity breaks monotony. More unexpected adventures, please." },
    { "input": "I was productive and crossed everything off my list.", "output": "Productivity satisfaction is real. You accomplished what you set out to." },
    { "input": "I said something witty and people laughed.", "output": "Being funny is gift and talent. You brought joy." },
    { "input": "I received a gift that was exactly right.", "output": "Perfect gifts show understanding. Feel how well you're known." },
    { "input": "I had a great day at work with no drama.", "output": "Drama-free work days are gifts. Appreciate the peace." },
    { "input": "I created something and I'm proud of it.", "output": "Creation brings pride. You made something from nothing." },
    { "input": "I reached out and someone was excited to hear from me.", "output": "Being wanted is affirming. Your presence matters to people." },
    { "input": "I felt inspired after reading something meaningful.", "output": "Good content shifts perspective. You let it in." },
    { "input": "I overcame inertia and got things done.", "output": "Momentum starts with first step. You took it." },
    { "input": "I noticed something about myself that's positive.", "output": "Self-awareness of strengths is empowering. Recognize your qualities." },
    { "input": "I had a moment of perfect peace.", "output": "Peace moments are precious. They remind us what's possible." },
    { "input": "I'm grateful for today in a simple way.", "output": "Simple gratitude is most powerful. This attitude compounds." },

    # STRUGGLES WITH SELF (546-595)
    { "input": "I'm struggling with self-worth today.", "output": "Worth struggles are normal but not truth. You are worthy regardless of productivity or approval." },
    { "input": "I feel like I'm not enough.", "output": "Insufficiency feelings are common lies we tell ourselves. You are enough as you are right now." },
    { "input": "I'm comparing myself to others and losing.", "output": "Comparison is the thief of joy. Focus on your own path, not their highlight reel." },
    { "input": "I feel small and unimportant.", "output": "Smallness feeling doesn't match reality. You matter more than your brain tells you." },
    { "input": "I'm doubting my abilities again.", "output": "Ability doubt cycles back. Remember what you've done before—you can do this too." },
    { "input": "I feel unlovable sometimes.", "output": "Unlovable feelings are depression lying. You are deeply lovable." },
    { "input": "I'm struggling to believe in myself.", "output": "Self-belief wavers. That's human. Find evidence of your capability when doubt hits." },
    { "input": "I feel like a burden to everyone.", "output": "Burden feeling often means you need to rest. You're not a burden—you're allowed to need support." },
    { "input": "I'm being too hard on myself.", "output": "Self-criticism over-adjusted. Give yourself the kindness you'd give a friend." },
    { "input": "I feel broken and unfixable.", "output": "Broken feelings are real but not final. You're not broken—you're healing." },
    { "input": "I don't trust my own judgment.", "output": "Judgment doubt happens. Gather data, consult trusted people, then trust yourself." },
    { "input": "I feel like I'm disappointing everyone.", "output": "Disappointment fear is often projection. You're probably doing better than you think." },
    { "input": "I'm not good at anything.", "output": "Everything-bad thinking is cognitive distortion. Name one thing you do well and build from there." },
    { "input": "I feel invisible and overlooked.", "output": "Invisibility feeling happens but check reality. Make yourself visible by showing up." },
    { "input": "I'm too sensitive and it's a flaw.", "output": "Sensitivity is often strength. The world needs your depth." },
    { "input": "I feel like a failure at life.", "output": "Failure feeling isn't fact. You've succeeded at many things—refocus there." },
    { "input": "I'm my own worst enemy.", "output": "Self-sabotage is real but stoppable. Notice when you do it and choose differently." },
    { "input": "I feel inadequate for this role.", "output": "Role inadequacy feelings hit most people. You were hired for a reason—trust that." },
    { "input": "I'm uninteresting and boring.", "output": "Boring feeling is subjective and usually depression talking. You're interesting to someone." },
    { "input": "I don't deserve good things.", "output": "Unworthiness belief is lie. You deserve good things just for existing." },
    { "input": "I feel like I'm faking it all.", "output": "Authenticity struggles are real. But fake-it-till-you-make-it works. Keep faking confidently." },
    { "input": "I can't do anything right.", "output": "Everything-wrong thinking is distortion. You do many things right—focus there." },
    { "input": "I'm not as good as I pretend to be.", "output": "Pretense anxiety is common. Actually, you're probably more capable than you think." },
    { "input": "I feel stuck and unchangeable.", "output": "Stuck feeling suggests need for change. You can change—start small." },
    { "input": "I'm too much for people.", "output": "Too-much feeling is about people's capacity, not your value. Find people who have capacity." },
    { "input": "I feel fundamentally flawed.", "output": "Fundamental flaw belief isn't true. All humans have imperfections—that's normalcy." },
    { "input": "I don't fit anywhere.", "output": "Not-fitting feeling means you haven't found your people yet. Keep looking." },
    { "input": "I'm ashamed of who I really am.", "output": "Real-self shame is painful but treatable. You're worthy of self-acceptance." },
    { "input": "I feel like I'm living a lie.", "output": "Authenticity gap creates this feeling. Start revealing your true self gradually." },
    { "input": "I'm exhausted from trying to be normal.", "output": "Normal-trying exhaustion is real. Give yourself permission to be weird. Weird is better." },

    # BODY & APPEARANCE (596-630)
    { "input": "I'm uncomfortable in my body today.", "output": "Body discomfort is temporary. Try movement, kindness, or rest—whatever feels good." },
    { "input": "I'm having negative thoughts about my appearance.", "output": "Appearance critical thoughts are loud but not fact. Counter them with kindness." },
    { "input": "I gained weight and I'm struggling with it.", "output": "Weight changes trigger feelings. Weight doesn't define worth. Your body is worthy at all sizes." },
    { "input": "I'm self-conscious about how I look.", "output": "Self-consciousness is normal. Remember most people are focused on themselves, not judging you." },
    { "input": "I feel ugly today.", "output": "Ugly feelings come and go. This feeling isn't fact. You're a work of art." },
    { "input": "I'm noticing my body aging and I feel sad.", "output": "Aging awareness is loss, but also wisdom. Grieve what changes while celebrating what remains." },
    { "input": "I feel good about how I look today.", "output": "Body confidence moments are to be celebrated. Let yourself feel attractive." },
    { "input": "I'm learning to appreciate my body's strength.", "output": "Function over form is empowering. Your body does amazing things." },
    { "input": "I bought clothes that make me feel good.", "output": "Feeling good in your skin matters. Invest in clothes that serve you." },
    { "input": "I'm trying to accept my body as is.", "output": "Body acceptance is journey. You're making progress with each compassionate moment." },
    { "input": "My scars tell my story and I'm okay with that.", "output": "Scars are evidence of survival. You've healed from things—that's beautiful." },
    { "input": "I'm embracing my natural body without filters.", "output": "Unfiltered self-acceptance is radical. You're authentic." },
    { "input": "I felt attractive and powerful today.", "output": "Feeling attractive shifts energy. Carry that feeling forward." },
    { "input": "I'm being kinder to my body.", "output": "Body kindness is revolution. Your body is home—treat it gently." },
    { "input": "I wore something bold and felt confident.", "output": "Bold choices shift how you feel. More boldness, please." },
    { "input": "I'm not waiting until I lose weight to live.", "output": "Life-living refusal to wait is wisdom. You deserve now, not conditional later." },
    { "input": "My body carries me through my life and I'm grateful.", "output": "Function gratitude is perspective. Your body works hard for you." },
    { "input": "I'm learning my body's limits and respecting them.", "output": "Limit respect prevents injury. You're listening to your body." },
    { "input": "I feel comfortable in my skin for the first time.", "output": "Skin comfort is liberation. You've arrived at acceptance." },
    { "input": "I'm dressing for myself, not for others.", "output": "Self-dressing is autonomy. Wear what makes you feel good." },
    { "input": "I accepted a compliment about my appearance.", "output": "Compliment acceptance is kindness to yourself and compliment-giver. Both win." },
    { "input": "I'm not retouching my photos anymore.", "output": "Filter freedom is authentic. Real you is better." },
    { "input": "I feel sexy and that feels good.", "output": "Sexual confidence is empowering. Own it." },
    { "input": "I'm learning my body type suits certain styles.", "output": "Personal style clarity helps. You're dressing for your shape." },
    { "input": "I danced freely without self-consciousness.", "output": "Movement joy without judgment is freedom. More of this." },

    # WORK & CAREER (631-680)
    { "input": "I had a great day at work and felt accomplished.", "output": "Work accomplishment is affirming. You contributed real value today." },
    { "input": "My boss gave me meaningful feedback.", "output": "Good feedback helps growth. Your boss is invested in your development." },
    { "input": "I led a meeting and people listened to me.", "output": "Being heard in work is validating. Your voice matters." },
    { "input": "I'm learning my role faster than expected.", "output": "Fast learning shows capability. You're settling in well." },
    { "input": "I solved a problem that was stumping the team.", "output": "Problem-solving contribution is valued. You brought real skill." },
    { "input": "I got positive feedback on my project.", "output": "Project praise is external validation of good work. Internalize this." },
    { "input": "I'm building good relationships with my colleagues.", "output": "Work relationships matter. You're creating bonds that help everyone." },
    { "input": "I found a work mentor who's guiding me.", "output": "Mentorship accelerates growth. Learn everything from this person." },
    { "input": "I'm becoming more confident in my expertise.", "output": "Expertise confidence compounds. Trust what you know." },
    { "input": "I finished a challenging project ahead of schedule.", "output": "Ahead-of-schedule delivery is impressive. You're efficient." },
    { "input": "I'm getting better at work-life balance.", "output": "Balance is possible with boundaries. You're protecting your time." },
    { "input": "I asked for what I needed at work and got it.", "output": "Asking-and-receiving is empowering. Advocacy works." },
    { "input": "I'm growing in my current role.", "output": "In-role growth deepens mastery. You're getting better at what you do." },
    { "input": "I have a project I'm really excited about.", "output": "Work excitement is fuel. Let that energy propel you forward." },
    { "input": "I'm respected by my team.", "output": "Team respect is earned. Your contributions are valued." },
    { "input": "I'm developing skills that interest me.", "output": "Interest-driven skill-building is motivating. Keep pursuing these." },
    { "input": "I had a productive one-on-one with my boss.", "output": "Good check-ins build relationships. Your boss sees you." },
    { "input": "I'm not dreading Monday anymore.", "output": "Monday shift from dread to neutral is progress. Keep building." },
    { "input": "I helped train someone new and felt useful.", "output": "Teaching others affirms your knowledge. You're a valuable resource." },
    { "input": "I'm in a better team dynamic now.", "output": "Better dynamics improve everything. You're in a good place." },
    { "input": "I got a client who appreciates my work.", "output": "Appreciative clients make work easier. You found good fit." },
    { "input": "I'm more strategic about my career development.", "output": "Career intentionality drives growth. You're in control." },
    { "input": "I realized I'm actually good at what I do.", "output": "Competence realization is empowering. You ARE good at this." },
    { "input": "I'm contributing to something meaningful.", "output": "Meaningful work fulfills. Your work has purpose." },
    { "input": "I negotiated something at work and won.", "output": "Workplace negotiation wins build confidence. You can advocate for yourself." },
    { "input": "I'm working on a passion project at work.", "output": "Passion at work is rare. You've found something you love." },
    { "input": "My hard work is being recognized.", "output": "Recognition validates effort. Your work is being seen." },
    { "input": "I'm mentoring others in my field.", "output": "Field mentorship spreads your impact. You're lifting others." },
    { "input": "I'm considering a career change and I'm excited.", "output": "Career excitement about change is good sign. Follow that excitement." },
    { "input": "I found work flexibility that helps my life.", "output": "Work flexibility transforms life. You're in better arrangement." },

    # SPIRITUAL & MEANING (681-730)
    { "input": "I felt a moment of spiritual peace today.", "output": "Spiritual moments connect us to something bigger. Let that feeling linger." },
    { "input": "I'm exploring my spirituality and it feels right.", "output": "Spiritual exploration is personal. Trust what resonates with you." },
    { "input": "I prayed or meditated and felt heard.", "output": "Spiritual practice creates connection. You're deepening yours." },
    { "input": "I found meaning in something unexpected.", "output": "Meaning in surprises is gift. You're seeing deeper than surface." },
    { "input": "I'm living in alignment with my values.", "output": "Values alignment is integrity. Your life reflects what matters." },
    { "input": "I experienced synchronicity and it felt magical.", "output": "Meaningful coincidences shift perspective. You're tuned in." },
    { "input": "I'm discovering my life purpose.", "output": "Purpose discovery is journey. You're moving in right direction." },
    { "input": "I felt connected to something greater than myself.", "output": "Transcendent connection heals. You felt bigger picture." },
    { "input": "I'm questioning beliefs that no longer serve me.", "output": "Belief examination is evolution. You're becoming more authentic." },
    { "input": "I found community with my spiritual practice.", "output": "Spiritual community grounds practice. You belong here." },
    { "input": "I'm being more intentional about my life.", "output": "Intentionality creates meaning. You're directing your life." },
    { "input": "I forgave myself for something I thought was unforgivable.", "output": "Self-forgiveness is spiritual work. You're healing deeply." },
    { "input": "I feel like my life has purpose now.", "output": "Purpose feeling is anchor. You've found meaning." },
    { "input": "I'm honoring my authentic spiritual path.", "output": "Authentic spirituality is personal. Your path is valid." },
    { "input": "I saw beauty in an everyday moment and felt awe.", "output": "Awe in everyday life is presence. You're awake to wonder." },
    { "input": "I'm living with more gratitude and it's transforming me.", "output": "Gratitude practice transforms perspective. You're seeing abundance." },
    { "input": "I made a decision aligned with my values.", "output": "Values-aligned decisions feel right. Trust that feeling." },
    { "input": "I'm releasing control and trusting the process.", "output": "Trust development is spiritual maturity. You're learning to surrender." },
    { "input": "I felt the presence of someone I've lost.", "output": "Spiritual connection to departed is real. They're with you." },
    { "input": "I'm building a life that honors who I am.", "output": "Honoring self is spiritual practice. Keep building authentically." },
    { "input": "I experienced divine timing in my life.", "output": "Timing magic happens when you're ready. You were prepared." },
    { "input": "I'm learning to trust my intuition as spiritual guidance.", "output": "Intuition as spiritual guidance is wisdom. Listen to it." },
    { "input": "I felt held by something I can't explain.", "output": "Unexplainable holding is spiritual. You're supported." },
    { "input": "I'm living my values more authentically.", "output": "Values embodiment is integrity. You're aligning inside and outside." },
    { "input": "I made peace with my past through spiritual practice.", "output": "Spiritual peace with past is liberation. You've healed." },
    { "input": "I'm drawn to deeper questions about meaning.", "output": "Depth questions are spiritual awakening. Explore them." },
    { "input": "I'm learning that I'm enough in the eyes of the divine.", "output": "Divine acceptance is absolute. You're worthy as you are." },
    { "input": "I experienced unconditional love in a moment.", "output": "Unconditional love moments transform. You've felt what matters most." },
    { "input": "I'm surrendering my timeline to something greater.", "output": "Surrender of control is faith. You're trusting now." },
    { "input": "I found my people who share my spiritual values.", "output": "Spiritual community alignment is grounding. You're home." }
]

In [6]:
# ===================== TRAINING EXAMPLE =====================
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Example dataset for thesis (truncated for brevity - full version in original notebook)


# Quick function to run training
def run_training_demo(variant: int = 1, num_epochs: int = config.num_epochs):
    """
    Quick demo training for thesis.
    Set num_epochs=1 for quick test, num_epochs=3+ for serious training
    """
    if variant == 1:
        print("\n🔵 VARIANT 1 TRAINING DEMO")
        print("=" * 70)
        projector, history = train_variant_1(
            train_pairs=example_pairs[:int(len(example_pairs)*0.9)],
            val_pairs=example_pairs[int(len(example_pairs)*0.9):],  # Skip validation for quick demo
            num_epochs=num_epochs,
        )
        return projector, history
    
    elif variant == 2:
        print("\n🟢 VARIANT 2 TRAINING DEMO")
        print("=" * 70)
        projector, history = train_variant_2(
            train_pairs=example_pairs[:int(len(example_pairs)*0.9)],
            val_pairs=example_pairs[int(len(example_pairs)*0.9):],  # Skip validation for quick demo
            num_epochs=num_epochs,
        )
        return projector, history

# ===================== INFERENCE EXAMPLE =====================

def run_inference_demo_with_untrained_weights():
    """
    Complete inference demo: All 3 scenarios
    """
    print("\n" + "=" * 70)
    print("INFERENCE DEMO: All Three Scenarios")
    print("=" * 70)
    
    # Test prompt and context
    test_prompt = "When I think about my future, I feel nervous and uncertain."
    
    # Initialize inference engine
    print("\n[1/4] Initializing inference engine...")
    engine = KVInjectionInference()
    
    # SCENARIO 1: Baseline (no injection)
    print("\n[2/4] SCENARIO 1: Baseline Generation (No KV Injection)")
    print("-" * 70)
    vanilla_output = engine.generate_vanilla(
        prompt=test_prompt,
        max_new_tokens=50,
        temperature=0.7
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {vanilla_output}\n")
    
    # SCENARIO 2: Variant 1 (with untrained projector for demo)
    print("[3/4] SCENARIO 2: Variant 1 (BasicKVProjector - Untrained)")
    print("-" * 70)
    print("Creating untrained Variant 1 projector...")
    projector_v1 = BasicKVProjector(
        num_emotions=config.num_emotions,
        num_heads=engine.decoder.config.num_attention_heads,  # Get from actual decoder
        num_kv_heads=engine.decoder.config.num_key_value_heads,  # Get from actual decoder (8, not 32)
        head_dim=engine.decoder.config.hidden_size // engine.decoder.config.num_attention_heads,
        prefix_len=config.prefix_len
    ).to(DEVICE)  # FIX: Move to device
    engine.projector_v1 = projector_v1  # Use untrained
    
    v1_output, v1_debug = engine.generate_with_variant_1(
        prompt=test_prompt,
        emotion_text=test_prompt,
        max_new_tokens=50,
        temperature=0.7,
        show_cache_info=True
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {v1_output}\n")
    
    # SCENARIO 3: Variant 2 (with untrained projector for demo)
    print("[4/4] SCENARIO 3: Variant 2 (ModulatedKVProjector - Untrained)")
    print("-" * 70)
    print("Creating untrained Variant 2 projector...")
    projector_v2 = ModulatedKVProjector(
        num_emotions=config.num_emotions,
        num_heads=engine.decoder.config.num_attention_heads,  # Get from actual decoder
        num_kv_heads=engine.decoder.config.num_key_value_heads,  # Get from actual decoder (8, not 32)
        head_dim=engine.decoder.config.hidden_size // engine.decoder.config.num_attention_heads,
        prefix_len=config.prefix_len
    ).to(DEVICE)  # FIX: Move to device
    engine.projector_v2 = projector_v2  # Use untrained
    
    v2_output, v2_debug = engine.generate_with_variant_2(
        prompt=test_prompt,
        emotion_text=test_prompt,
        max_new_tokens=50,
        temperature=0.7,
        show_cache_info=True
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {v2_output}")
    print(f"Head gates (first 8): {v2_debug['head_gates'][:8]}\n")
    
    print("=" * 70)
    print("COMPARISON SUMMARY")
    print("=" * 70)
    print(f"\n1️⃣  BASELINE: {vanilla_output[:100]}...")
    print(f"\n2️⃣  VARIANT 1: {v1_output[:100]}...")
    print(f"\n3️⃣  VARIANT 2: {v2_output[:100]}...")

def run_inference_demo_with_trained_weights():
    """
    Complete inference demo: All 3 scenarios
    """
    print("\n" + "=" * 70)
    print("INFERENCE DEMO: All Three Scenarios")
    print("=" * 70)
    
    test_prompt = "When I think about my future, I feel nervous and uncertain."
    
    # Initialize inference engine
    print("\n[1/4] Initializing inference engine...")
    engine = KVInjectionInference()
    
    # SCENARIO 1: Baseline (no injection)
    print("\n[2/4] SCENARIO 1: Baseline Generation (No KV Injection)")
    print("-" * 70)
    vanilla_output = engine.generate_vanilla(
        prompt=test_prompt,
        max_new_tokens=250,
        temperature=0.7
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {vanilla_output}\n")
    
    # SCENARIO 2: Variant 1 (with TRAINED projector)
    print("[3/4] SCENARIO 2: Variant 1 (BasicKVProjector - TRAINED)")
    print("-" * 70)
    variant1_path = os.path.join(config.checkpoint_dir, "variant1_projector.pt")
    if engine.load_projector_v1(variant1_path):
        print("✓ Trained weights loaded")
    else:
        print("⚠ Trained weights not found, using untrained")
        projector_v1 = BasicKVProjector(
            num_emotions=config.num_emotions,
            num_heads=engine.decoder.config.num_attention_heads,
            num_kv_heads=engine.decoder.config.num_key_value_heads,
            head_dim=engine.decoder.config.hidden_size // engine.decoder.config.num_attention_heads,
            prefix_len=config.prefix_len
        ).to(DEVICE)
        engine.projector_v1 = projector_v1
    
    v1_output, v1_debug = engine.generate_with_variant_1(
        prompt=test_prompt,
        emotion_text=test_prompt,
        max_new_tokens=250,
        temperature=0.7,
        show_cache_info=True
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {v1_output}\n")
    
    # SCENARIO 3: Variant 2 (with TRAINED projector)
    print("[4/4] SCENARIO 3: Variant 2 (ModulatedKVProjector - TRAINED)")
    print("-" * 70)
    variant2_path = os.path.join(config.checkpoint_dir, "variant2_projector.pt")
    if engine.load_projector_v2(variant2_path):
        print("✓ Trained weights loaded")
    else:
        print("⚠ Trained weights not found, using untrained")
        projector_v2 = ModulatedKVProjector(
            num_emotions=config.num_emotions,
            num_heads=engine.decoder.config.num_attention_heads,
            num_kv_heads=engine.decoder.config.num_key_value_heads,
            head_dim=engine.decoder.config.hidden_size // engine.decoder.config.num_attention_heads,
            prefix_len=config.prefix_len
        ).to(DEVICE)
        engine.projector_v2 = projector_v2
    
    v2_output, v2_debug = engine.generate_with_variant_2(
        prompt=test_prompt,
        emotion_text=test_prompt,
        max_new_tokens=250,
        temperature=0.7,
        show_cache_info=True
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {v2_output}")
    print(f"Head gates (first 8): {v2_debug['head_gates'][:8]}\n")
    
    print("=" * 70)
    print("COMPARISON SUMMARY")
    print("=" * 70)
    print(f"\n1️⃣  BASELINE: {vanilla_output[:100]}...")
    print(f"\n2️⃣  VARIANT 1: {v1_output[:100]}...")
    print(f"\n3️⃣  VARIANT 2: {v2_output[:100]}...")
    
    # EVALUATION: Emotional Alignment
    print("\n" + "=" * 70)
    print("EMOTIONAL ALIGNMENT EVALUATION")
    print("=" * 70)
    
    # Variant 1 Evaluation
    print("\n📊 VARIANT 1 (BasicKVProjector) - Emotional Alignment:")
    sim_vanilla_v1, sim_kv_v1, delta_v1 = compute_emotional_alignment(
        input_text=test_prompt,
        vanilla_output=vanilla_output,
        kv_output=v1_output,
        emotion_extractor=engine.emotion_extractor,
        encoder_tokenizer=engine.encoder_tokenizer,
        device=DEVICE
    )
    print(f"  Input vs Vanilla Output:     {sim_vanilla_v1:.4f}")
    print(f"  Input vs Variant 1 Output:   {sim_kv_v1:.4f}")
    print(f"  Improvement (Delta):         {delta_v1:+.4f}")
    
    # Variant 2 Evaluation
    print("\n📊 VARIANT 2 (ModulatedKVProjector) - Emotional Alignment:")
    sim_vanilla_v2, sim_kv_v2, delta_v2 = compute_emotional_alignment(
        input_text=test_prompt,
        vanilla_output=vanilla_output,
        kv_output=v2_output,
        emotion_extractor=engine.emotion_extractor,
        encoder_tokenizer=engine.encoder_tokenizer,
        device=DEVICE
    )
    print(f"  Input vs Vanilla Output:     {sim_vanilla_v2:.4f}")
    print(f"  Input vs Variant 2 Output:   {sim_kv_v2:.4f}")
    print(f"  Improvement (Delta):         {delta_v2:+.4f}")
    
    print("\n" + "=" * 70)
# Run demo (uncomment to execute)
"""
projector1, history1 = run_training_demo(variant=1)

plot_training_history(
    history1,
    title="Variant 1 – Basic KV Projector Training",
    save_path="variant1_loss.png"
)
# 44m 50.7s
"""
"""
projector2, history2 = run_training_demo(variant=2)

plot_training_history(
    history2,
    title="Variant 2 – Head-Wise Modulated KV Projector",
    save_path="variant2_loss.png"
)
# 33m 42.0s
"""

#run_inference_demo_with_trained_weights()


# ===================== COMPLETE PIPELINE: INFERENCE + EVALUATION =====================

def run_inference_and_evaluate():
    """
    Complete pipeline: Generate outputs with inference, then evaluate emotional alignment.
    
    This function combines:
    1. run_inference_demo_with_trained_weights() - generates vanilla, variant1, variant2 outputs
    2. compute_emotional_alignment() - measures how well KV injection preserves input emotion
    
    Output:
    - Inference results (vanilla, V1, V2 outputs)
    - Emotional alignment metrics comparing vanilla vs KV-injected outputs
    """
    print("\n" + "=" * 70)
    print("COMPLETE PIPELINE: INFERENCE + EMOTIONAL ALIGNMENT EVALUATION")
    print("=" * 70)
    
    test_prompt = "When I think about my future, I feel nervous and uncertain."
    
    # ===== STEP 1: INFERENCE =====
    print("\n[STEP 1/2] Running Inference with Trained Weights...")
    print("-" * 70)
    
    # Initialize inference engine
    engine = KVInjectionInference()
    
    # SCENARIO 1: Baseline (no injection)
    print("\n[1/3] SCENARIO 1: Baseline Generation (No KV Injection)")
    vanilla_output = engine.generate_vanilla(
        prompt=test_prompt,
        max_new_tokens=250,
        temperature=0.7
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {vanilla_output}\n")
    
    # SCENARIO 2: Variant 1 (with trained projector)
    print("[2/3] SCENARIO 2: Variant 1 (BasicKVProjector - TRAINED)")
    print("-" * 70)
    variant1_path = os.path.join(config.checkpoint_dir, "variant1_projector.pt")
    if engine.load_projector_v1(variant1_path):
        print("✓ Trained weights loaded")
    else:
        print("⚠ Trained weights not found, using untrained")
        projector_v1 = BasicKVProjector(
            num_emotions=config.num_emotions,
            num_heads=engine.decoder.config.num_attention_heads,
            num_kv_heads=engine.decoder.config.num_key_value_heads,
            head_dim=engine.decoder.config.hidden_size // engine.decoder.config.num_attention_heads,
            prefix_len=config.prefix_len
        ).to(DEVICE)
        engine.projector_v1 = projector_v1
    
    v1_output, v1_debug = engine.generate_with_variant_1(
        prompt=test_prompt,
        emotion_text=test_prompt,
        max_new_tokens=250,
        temperature=0.7,
        show_cache_info=False
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {v1_output}\n")
    
    # SCENARIO 3: Variant 2 (with trained projector)
    print("[3/3] SCENARIO 3: Variant 2 (ModulatedKVProjector - TRAINED)")
    print("-" * 70)
    variant2_path = os.path.join(config.checkpoint_dir, "variant2_projector.pt")
    if engine.load_projector_v2(variant2_path):
        print("✓ Trained weights loaded")
    else:
        print("⚠ Trained weights not found, using untrained")
        projector_v2 = ModulatedKVProjector(
            num_emotions=config.num_emotions,
            num_heads=engine.decoder.config.num_attention_heads,
            num_kv_heads=engine.decoder.config.num_key_value_heads,
            head_dim=engine.decoder.config.hidden_size // engine.decoder.config.num_attention_heads,
            prefix_len=config.prefix_len
        ).to(DEVICE)
        engine.projector_v2 = projector_v2
    
    v2_output, v2_debug = engine.generate_with_variant_2(
        prompt=test_prompt,
        emotion_text=test_prompt,
        max_new_tokens=250,
        temperature=0.7,
        show_cache_info=False
    )
    print(f"Prompt: {test_prompt}")
    print(f"Output: {v2_output}\n")
    
    # ===== STEP 2: EMOTIONAL ALIGNMENT EVALUATION =====
    print("\n" + "=" * 70)
    print("[STEP 2/2] Computing Emotional Alignment Metrics...")
    print("=" * 70)
    
    # Variant 1 Evaluation
    print("\n📊 VARIANT 1 (BasicKVProjector) - Emotional Alignment:")
    results = compute_emotional_alignment(
        input_text=test_prompt,
        vanilla_output=vanilla_output,
        v1_output=v1_output,
        v2_output=v2_output,
        emotion_extractor=engine.emotion_extractor,
        encoder_tokenizer=engine.encoder_tokenizer,
        device=DEVICE
    )
    sim_v1 = results["sim_values"][0]
    delta_v1 = results["delta_values"][0]
    sim_vanilla_v1 = results["sim_values"][0] - delta_v1
    
    print(f"  Input vs Vanilla Output:     {sim_vanilla_v1:.4f}")
    print(f"  Input vs Variant 1 Output:   {sim_v1:.4f}")
    print(f"  Improvement (Delta):         {delta_v1:+.4f}")
    if delta_v1 > 0:
        print(f"  ✓ Variant 1 preserves emotion BETTER than vanilla ({delta_v1*100:.2f}% improvement)")
    else:
        print(f"  ✗ Variant 1 preserves emotion WORSE than vanilla ({delta_v1*100:.2f}% degradation)")
    
    # Variant 2 Evaluation
    print("\n📊 VARIANT 2 (ModulatedKVProjector) - Emotional Alignment:")
    sim_v2 = results["sim_values"][1]
    delta_v2 = results["delta_values"][1]
    sim_vanilla_v2 = results["sim_values"][1] - delta_v2
    
    print(f"  Input vs Vanilla Output:     {sim_vanilla_v2:.4f}")
    print(f"  Input vs Variant 2 Output:   {sim_v2:.4f}")
    print(f"  Improvement (Delta):         {delta_v2:+.4f}")
    if delta_v2 > 0:
        print(f"  ✓ Variant 2 preserves emotion BETTER than vanilla ({delta_v2*100:.2f}% improvement)")
    else:
        print(f"  ✗ Variant 2 preserves emotion WORSE than vanilla ({delta_v2*100:.2f}% degradation)")
    
    # ===== FINAL SUMMARY =====
    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    print(f"\n🎯 Best Performer:")
    if delta_v1 > delta_v2:
        print(f"   Variant 1 (BasicKVProjector) with {delta_v1:+.4f} improvement")
    elif delta_v2 > delta_v1:
        print(f"   Variant 2 (ModulatedKVProjector) with {delta_v2:+.4f} improvement")
    else:
        print(f"   Both variants tied at {delta_v1:+.4f} improvement")
    
    print(f"\n📈 Improvement Summary:")
    print(f"   Variant 1 Delta: {delta_v1:+.4f}")
    print(f"   Variant 2 Delta: {delta_v2:+.4f}")
    print(f"   Difference (V2 - V1): {delta_v2 - delta_v1:+.4f}")
    
    print("\n" + "=" * 70)
    
    return {
        "vanilla_output": vanilla_output,
        "v1_output": v1_output,
        "v2_output": v2_output,
        "v1_metrics": {"sim_vanilla": sim_vanilla_v1, "sim_v1": sim_v1, "delta": delta_v1},
        "v2_metrics": {"sim_vanilla": sim_vanilla_v2, "sim_v2": sim_v2, "delta": delta_v2}
    }


# Run complete pipeline (uncomment to execute)
results = run_inference_and_evaluate()
print(f"Results: {results}")


COMPLETE PIPELINE: INFERENCE + EMOTIONAL ALIGNMENT EVALUATION

[STEP 1/2] Running Inference with Trained Weights...
----------------------------------------------------------------------
Initializing KV Injection Inference Engine...

  Loading decoder...


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

  ✓ Decoder: microsoft/Phi-4-mini-instruct
  Loading emotion extractor...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Emotion extractor: distilbert-base-uncased (pre-trained)

[1/3] SCENARIO 1: Baseline Generation (No KV Injection)
Prompt: When I think about my future, I feel nervous and uncertain.
Output: When I think about my future, I feel nervous and uncertain. I don’t know what I want to do, and I’m scared of making the wrong choice. How can I overcome these feelings and gain confidence in my decision-making abilities?

1. Acknowledge Your Feelings: Recognize and accept that feeling nervous and uncertain about the future is completely normal. Accepting these emotions can help you move forward.

2. Self-Reflection: Spend time reflecting on your interests, passions, and strengths. Consider what activities make you feel energized and engaged. This can provide clarity on what you may want to pursue.

3. Set Small Goals: Start by setting small, achievable goals to gain confidence in your decision-making abilities. This can help build momentum and provide a sense of accomplishment.

4. Seek Guidanc

TypeError: EmotionExtractor.forward() got an unexpected keyword argument 'token_type_ids'